# GeoNUSAF Part 2 / Stage 4 — TC-SegFormer, real vs real+synthetic (R0 / R1)

**Block fold 1. One arm per session. Change `ARM` in section 2 and run top to bottom.**

| arm | train pairs | composition |
|---|---|---|
| `R0` | 804 | 804 real fold-1 train pairs |
| `R1` | 1608 | the same 804 real + 804 synthetic from `fake_pairs_R1/` |

Validation is the **136 real fold-1 tiles (408 images) in both arms**, never a synthetic pixel.

This is the TC-SegFormer arm of the Stage-4 table. It carries the proposed model over from
`5__tc-segformer.ipynb` unchanged — residual H/2 detail path, CSA reliability reweighting,
CE x CSA + CSA-masked Lovasz + soft-clDice, and the class-presence-balanced sampler — and runs it
inside the **same harness** as `geonusaf-stage4-segformer-b0-R0R1.ipynb` and
`geonusaf-stage4-unetformer-r18-R0R1.ipynb`, so the three architectures' R1-R0 deltas are
comparable to each other.

## Five deliberate departures from `5__tc-segformer.ipynb`

**1. The splitter is the generators'.** Part 1's `split_block` drops a 2-tile buffer around *every*
chunk edge, giving **260** train tiles at fold 1 (`train_sha1 63b87833f4b4`). Stages 1, 2 and 3 —
and the other two Stage-4 notebooks — drop the buffer only around the **val** block, giving **268**
tiles (`eb0aebdd9919`). The synthetic pairs were generated from that 268-tile set. Using part 1's
splitter here would leave 8 tiles in the S4 train set that the generator did see, and the arms would
no longer line up with the B0 and UNetFormer rows. Section 8 asserts the generators' sha1.

**2. Step-based, not epoch-based.** R1's training pool is twice R0's, so a pass is 100 steps against
R0's 50. Holding `MAX_EPOCHS` fixed would hand R1 twice the gradient updates and the result would
measure compute, not data. Both arms get `MAX_STEPS = 6000` with a cosine horizon of 6000 — at R0
that is 120 passes, the part-1 horizon. The two **loss schedules** move with it: the Lovasz ramp
becomes 500 steps (part 1's 10 epochs) and clDice switches on at step 1000 (part 1's epoch 20).

**3. The augmentation bug is fixed.** Part 1 seeded a fresh `RandomState` from `self.epoch`, the main
process called `train_ds.set_epoch(epoch)`, and `train_dl` ran `persistent_workers=True` at
`NUM_WORKERS=2`. Persistent workers hold their own copy of the dataset from the first iterator, so
`self.epoch` never changed inside them and **every pass replayed identical rot90 and jitter draws**.
This is the same defect the B0 notebook had. Augmentation here draws from the worker-local
`np.random` stream, which advances on every call regardless of worker lifetime. Section 11 asserts it.

**4. Determinism off.** `use_deterministic_algorithms(True, warn_only=True)` forces a sort-based
scatter backward that costs roughly 6x on this workload. Both arms run without it.

**5. `refine`, not `classifier`, in the standalone loader.** Part 1's section 23 rebuilt the H/2
branch's final conv as `s.classifier` while the trained model names it `self.refine`, so
`load_state_dict(strict=True)` on any real checkpoint raised `Missing key(s): refine.weight`.
Fixed in section 23 here.

Consequence: **the R0 number this notebook produces will not equal the published 0.5582.** That run
had a 260-tile train set, frozen augmentation, determinism on and an epoch-based schedule. R0 and R1
must come out of the same harness or the delta means nothing. Do not mix the part-1 fold-1 number
into the Stage-4 table.

## Prerequisites
- Internet **ON**. Secrets: `HF_TOKEN` (write) and `WANDB_API_KEY`.
- Inputs attached: `sugam24/satellite-datasets` **and** `sugam24/geonusaf-fake-pairs-block-fold1`.
- GPU T4 x1. Budget ~3-4 h per arm (R1 is not slower per step, but early stopping lands later);
  resume from `last.pt` is automatic if a session dies.

## 1. Environment

In [ ]:
import subprocess, sys
def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
# transformers is pinned below 5.0 on purpose: 5.x renamed decode_head.linear_c to
# linear_projections, which changes every checkpoint key and would silently invalidate the
# part-1 and Stage-4 checkpoints. Section 13 carries a shim so the forward pass survives
# either name, but the pin keeps the state-dict keys stable.
pipq("transformers>=4.40,<5", "huggingface_hub>=0.23", "wandb", "fvcore")

import os, re, gc, json, math, time, random, hashlib, warnings, shutil, copy
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import cv2
cv2.setNumThreads(0)   # critical: cv2 threads fight DataLoader workers on Kaggle's 4 vCPUs
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

import transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| cuda", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## 2. Configuration

`ARM` is the only field you change between sessions. Everything else is identical across the two
runs by construction — that is what makes the delta attributable to the data.

The schedule is expressed in **steps**, and so are the two loss ramps. `COS_HORIZON_STEPS ==
MAX_STEPS` so the cosine lands at zero exactly when training ends.

`CSA_ON_FAKE` is the one genuinely new decision. CSA's premise is that a band around each polygon
boundary carries annotator error and should be discounted. That premise does not obviously transfer
to a synthetic pair: the mask came first and the image was *painted from it*, so the boundary is
where the generator put it and the label there is correct by construction. Down-weighting it
discards supervision that is not actually noisy. The counter-argument is that the painted image is
blurry and ambiguous exactly at those boundaries, so the pixels are unreliable for a different
reason, and that keeping one training rule for every row is the cleaner protocol claim. The default
is **on**, for that uniformity; `CSA_ON_FAKE=False` is the ablation and section 9 prints the mean
CSA weight per class for real and synthetic side by side so the choice is made with numbers in view.

In [ ]:
@dataclass
class CFG:
    # ---- THE ONLY FIELD YOU CHANGE BETWEEN SESSIONS ----
    ARM: str = "R0"                    # "R0" (real only) | "R1" (real + 804 synthetic)

    # ---- run selection: fixed, the generators are fold-1 block models ----
    SPLIT_MODE: str = "block"
    FOLD: int = 1

    # ---- identity ----
    MODEL_NAME: str = "tcsegformer"
    ARCH_VERSION: str = "v2-residual"  # bump whenever the module layout changes; guards HF resume
    RESUME: bool = True                # False forces a fresh start regardless of the Hub
    BACKBONE: str = "nvidia/segformer-b0-finetuned-ade-512-512"

    # ---- data ----
    DATA_ROOT: Optional[str] = None    # None -> auto-detect under /kaggle/input
    FAKE_ROOT: Optional[str] = None    # None -> auto-detect fake_pairs_R1 under /kaggle/input
    CACHE_DIR: str = "/kaggle/working/cache512"
    PREBUILT_CACHE: str = ""           # a saved part-1 cache512 dataset skips the 25-40 min repack
    IMG_SIZE: int = 512
    NUM_CLASSES: int = 6
    IGNORE_INDEX: int = 255
    PRELOAD_RAM: bool = True           # ~1.6 GB real + ~1.05 GB synthetic

    # ---- split ----
    N_FOLDS: int = 3
    BLOCK_BUFFER: int = 2
    # sha1 of the fold-1 train id list, printed by Stage 1/2/3 section 8. Asserted, not eyeballed:
    # if S4 trains on a different real split than the generators saw, the whole experiment is void.
    UPSTREAM_TRAIN_SHA1: str = "eb0aebdd991971c319fc5fde92f8ad4aa3d82b65"

    # ---- synthetic mixing ----
    N_FAKE: int = 804                  # size of fake_pairs_R1, asserted on load
    # "real"  -> class weights from the 804 real pairs, IDENTICAL in both arms (default)
    # "mixed" -> recomputed over real+fake, which makes the loss a second moving variable
    WEIGHTS_FROM: str = "real"
    # same idea for the sampler's tile-presence frequency table: "real" keeps the per-record
    # weighting RULE identical across arms and lets the pool composition do the work
    SAMPLER_FREQ_FROM: str = "real"
    CSA_ON_FAKE: bool = True           # see section 2 prose; False is the ablation
    # Real records already carry hf/vf as separate rows; synthetic pairs do not. Leaving this off
    # keeps the augmentation policy per-record and identical for every training row.
    AUG_FLIP_FAKE: bool = False

    # ---- D: coarse-supervision-aware reliability weights (unchanged from part 1) ----
    CSA_ENABLE: bool = True
    CSA_TAU: tuple = (0.60, 0.35, 0.35, 0.60, 0.60, 0.60)
    CSA_DMAX_M: float = 6.0            # hard cap on the unreliable band, in nominal tile metres
    TILE_METRES: float = 300.0         # KEPT AT THE PART-1 VALUE ON PURPOSE -- see section 6
    CSA_WMIN: float = 0.25
    CSA_LOVASZ_THRESH: float = 0.60    # pixels below this are IGNORED by Lovasz (hard mask)
    CORE_THRESH: float = 0.999         # w >= this defines the "core" evaluation region

    # ---- B: thin-structure detail path ----
    USE_DETAIL: bool = True
    DETAIL_CH: tuple = (8, 16, 32)
    FUSE_CH: int = 64
    DETAIL_GATE_INIT: float = 0.1
    DROP_PATH: float = 0.1

    # ---- C: loss, with the epoch ramps converted to steps ----
    EN_BETA_AUTO: bool = True
    W_CAP: float = 10.0
    LABEL_SMOOTH: float = 0.05
    LOVASZ_LAMBDA: float = 0.75
    LOVASZ_WARMUP_STEPS: int = 500     # part 1: 10 epochs x ~50 steps
    CLDICE_ENABLE: bool = True
    CLDICE_MU: float = 0.30
    CLDICE_START_STEP: int = 1000      # part 1: epoch 20
    CLDICE_CLASSES: tuple = (1, 2)     # Road, River
    CLDICE_ITERS: int = 5
    CLDICE_RES: int = 256
    BALANCED_SAMPLER: bool = True
    SAMPLER_CAP: float = 8.0

    # ---- optimization: STEP-based, so R0 and R1 get identical compute ----
    BATCH_SIZE: int = 16
    GRAD_ACCUM: int = 1
    LR_HEAD: float = 6e-5
    LR_ENCODER: float = 6e-6
    LR_NEW: float = 6e-4               # randomly-initialised modules: 10x the pretrained head
    WEIGHT_DECAY: float = 1e-4
    MAX_STEPS: int = 6000              # == 120 passes at R0's 50 steps/pass
    COS_HORIZON_STEPS: int = 6000
    WARMUP_STEPS: int = 150
    VAL_EVERY_STEPS: int = 200
    PATIENCE_VALS: int = 8             # 8 x 200 = 1600 steps without improvement
    CLIP_NORM: float = 1.0
    EMA: bool = True
    EMA_DECAY: float = 0.999

    # ---- augmentation ----
    AUG_ROT90: bool = True
    AUG_PHOTOMETRIC: bool = True
    AUG_BRIGHTNESS: float = 0.10
    AUG_CONTRAST: float = 0.10

    # ---- evaluation ----
    BOOTSTRAP_N: int = 2000            # resamples over the 136 val base tiles

    # ---- runtime ----
    SEED: int = 42
    NUM_WORKERS: int = 4
    AMP: bool = True
    MAX_HOURS: float = 11.0            # stop and push state before Kaggle's 12 h cap

    # ---- persistence ----
    HF_USER: str = "sugam24"
    HF_PREFIX: str = "geonusaf-s4-tcsegformer"
    HF_PUSH: bool = True
    PUSH_LAST_EVERY_VALS: int = 5      # full-state pushes, in validations
    WANDB: bool = True
    WANDB_PROJECT: str = "geonusaf-stage4"
    OUT_DIR: str = "/kaggle/working/out"

cfg = CFG()
Path(cfg.OUT_DIR).mkdir(parents=True, exist_ok=True)
Path(cfg.CACHE_DIR).mkdir(parents=True, exist_ok=True)
assert cfg.ARM in ("R0", "R1"), cfg.ARM
USE_FAKE = (cfg.ARM == "R1")

CLASS_NAMES = ["Residential", "Road", "River", "Forest", "UnusedLand", "Agricultural"]
RUN_NAME = f"{cfg.HF_PREFIX}-{cfg.ARM}-{cfg.SPLIT_MODE}-fold{cfg.FOLD}"
HF_REPO  = f"{cfg.HF_USER}/{RUN_NAME}"
tag = f"{cfg.ARM}_{cfg.SPLIT_MODE}_fold{cfg.FOLD}"

print("RUN:", RUN_NAME, "| HF repo:", HF_REPO)
print(f"arm {cfg.ARM}: real 804 + synthetic {cfg.N_FAKE if USE_FAKE else 0}")
print(f"schedule: {cfg.MAX_STEPS} steps | warmup {cfg.WARMUP_STEPS} | cosine to "
      f"{cfg.COS_HORIZON_STEPS} | val every {cfg.VAL_EVERY_STEPS} | patience "
      f"{cfg.PATIENCE_VALS} vals")
print(f"loss ramps: lovasz 0->{cfg.LOVASZ_LAMBDA} over {cfg.LOVASZ_WARMUP_STEPS} steps | "
      f"clDice mu={cfg.CLDICE_MU} from step {cfg.CLDICE_START_STEP}")

## 3. Determinism — deliberately OFF

Part 1 ran `use_deterministic_algorithms(True, warn_only=True)`, which forces a sort-based scatter
backward that costs roughly 6x on this workload. It buys nothing here: both arms share the seed, the
split and the schedule, so run-to-run kernel nondeterminism is common-mode noise on a comparison,
not a confound. `cudnn.benchmark` is on instead, since the input shape never varies.

In [ ]:
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

def worker_init_fn(worker_id):
    # each worker gets its OWN np.random stream, and that stream advances on every augmentation
    # draw. This is what makes augmentation live across passes -- see section 11.
    s = (cfg.SEED * 7919 + worker_id) % (2**31)
    np.random.seed(s); random.seed(s)

set_seed(cfg.SEED)
g = torch.Generator(); g.manual_seed(cfg.SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("seeded:", cfg.SEED, "| deterministic algorithms: OFF | cudnn.benchmark: ON | device", DEVICE)

## 4. Filename parsing and base-tile grouping

Verbatim from part 1, including the 1224 / 408 asserts. `DATA_ROOT` is auto-detected because the
Kaggle mount path differs between accounts and hard-coding it has already cost a session once.

Records are sorted by `(tile, variant)` and that ordering **is** the row index into the cache
arrays, so the cache and `TILE_OF` can never drift apart. Section 6 asserts the stored ordering
against this one.

In [ ]:
def find_data_root(explicit):
    if explicit and Path(explicit, "images").is_dir(): return explicit
    for p in Path("/kaggle/input").rglob("satellite_dataset"):
        if (p / "images").is_dir() and (p / "masks").is_dir(): return str(p)
    raise FileNotFoundError("satellite_dataset not found -- attach sugam24/satellite-datasets")

DATA_ROOT = find_data_root(cfg.DATA_ROOT)
IMG_DIR = Path(DATA_ROOT) / "images"
MSK_DIR = Path(DATA_ROOT) / "masks"
print("DATA_ROOT:", DATA_ROOT)

PAT = re.compile(r"^image(\d+)(hf|vf)?\.png$")
records, unparsed = [], []
for p in sorted(IMG_DIR.iterdir()):
    m = PAT.match(p.name)
    if not m:
        unparsed.append(p.name); continue
    n, var = int(m.group(1)), (m.group(2) or "base")
    mask_p = MSK_DIR / p.name.replace("image", "mask", 1)
    records.append(dict(stem=p.stem, tile=n, variant=var, img=str(p), msk=str(mask_p)))

assert not unparsed, f"unparsed filenames: {unparsed[:10]}"
assert len(records) == 1224, f"expected 1224 files, got {len(records)}"
missing = [r["stem"] for r in records if not Path(r["msk"]).exists()]
assert not missing, f"missing masks: {missing[:10]}"

BASE_TILES = sorted({r["tile"] for r in records})
assert len(BASE_TILES) == 408, f"expected 408 base tiles, got {len(BASE_TILES)}"
from collections import Counter
assert set(Counter(r["tile"] for r in records).values()) == {3}, "every base tile needs 3 variants"

records.sort(key=lambda r: (r["tile"], {"base": 0, "hf": 1, "vf": 2}[r["variant"]]))
STEMS = [r["stem"] for r in records]
N_REAL = len(records)
TILE_OF_REAL = np.array([r["tile"] for r in records], dtype=np.int32)
print(f"{N_REAL} files | {len(BASE_TILES)} base tiles | 3 variants each | "
      f"ids {BASE_TILES[0]}..{BASE_TILES[-1]}")

## 5. Mask colour diagnostic and BGR channel-order assert

`(128,0,0)` = Residential and `(0,0,128)` = River are exact channel reverses. If BGR/RGB inverted,
the dominant class and the rarest class would swap silently and every number below would still look
plausible. The only reliable detector is a **count** check: Residential must vastly outnumber River.

In [ ]:
BGR_TO_CLASS = {
    (128, 0, 0): 0, (0, 128, 0): 1, (0, 0, 128): 2,
    (0, 128, 128): 3, (128, 128, 0): 4, (128, 0, 128): 5,
}
LUT = np.full((256, 256, 256), 255, dtype=np.uint8)
for (b, gg, r), c in BGR_TO_CLASS.items():
    LUT[b, gg, r] = c

def bgr_to_index(m_bgr):
    return LUT[m_bgr[:, :, 0], m_bgr[:, :, 1], m_bgr[:, :, 2]]

rng = np.random.RandomState(cfg.SEED)
probe = [records[i] for i in rng.choice(len(records), 6, replace=False)]
tot = np.zeros(7, dtype=np.int64)
NATIVE_SHAPE = None
for r in probe:
    m = cv2.imread(r["msk"], cv2.IMREAD_COLOR)
    assert m is not None, r["msk"]
    NATIVE_SHAPE = m.shape
    idx = bgr_to_index(m)
    u, c = np.unique(idx, return_counts=True)
    for uu, cc in zip(u, c):
        tot[6 if uu == 255 else uu] += cc
    print(f'{r["stem"]:>14}  shape={m.shape}')

print("\npixel totals over probe:", {n: int(v) for n, v in zip(CLASS_NAMES + ["ignore"], tot)})
assert tot[0] > 10 * max(tot[2], 1), (
    "CHANNEL ORDER HAZARD: Residential is not >> River. BGR/RGB is probably inverted. STOP.")
print("channel-order check PASSED: BGR confirmed, label values subset of {0..5, 255}")

## 6. One-time repack to a 512 cache — images, masks, **and CSA reliability weights**

Unchanged from part 1. Three memmapped `.npy` arrays, written once and skipped afterwards via the
marker file or `PREBUILT_CACHE`. **If you saved `cache512` as a Kaggle dataset from a part-1 run,
point `PREBUILT_CACHE` at it** — it saves 25-40 min, and there are two sessions here.

| Array | Kernel | Why |
|---|---|---|
| `images.npy` (1224,512,512,3) u8 | `INTER_AREA` | correct kernel for 4.6x downsampling |
| `masks.npy` (1224,512,512) u8 | LUT at **full res**, then `INTER_NEAREST` | resizing the RGB mask first would interpolate between class colours and manufacture phantom classes at every polygon edge |
| `relw.npy` (1224,512,512) u8 | CSA weight at **full res**, then `INTER_AREA` | the weight field is continuous, so area-averaging is correct |

The CSA weight: boundary set `B` from 4-neighbour label changes (image border excluded), `D` the
Euclidean distance transform of `~B`, per connected component `R = 2*mean(D)` as the local
half-width, `band = clip(tau_c * R, 1 px, DMAX_PX)`, `w = clip(D/band, w_min, 1)`.

**A note the paper needs.** `TILE_METRES = 300.0` is the superseded tile-size assumption; the
recovered scale is 703.5 m (0.300 m/px native). It is **kept at 300.0 here on purpose** so that the
CSA field is bit-identical to the one part 1 trained on — changing it would make S4 a different
model, not a data experiment. The consequence is that `CSA_DMAX_M = 6.0` is a nominal number: the
band it actually produces is `6.0 * 2345/300 = 46.9` native px, which at 0.300 m/px is **14.07
ground metres**. Quote 14.07 m in the paper, not 6 m. The cap stays inert for Road (`tau*R` is about
2.0 m) and binding for the area classes, so the intended asymmetry survives the correction.

In [ ]:
from scipy import ndimage as ndi

def csa_weight(idx, taus, wmin, dmax_px):
    lab = idx.astype(np.int16)
    b = np.zeros(lab.shape, dtype=bool)
    d = lab[:-1, :] != lab[1:, :]; b[:-1, :] |= d; b[1:, :] |= d
    d = lab[:, :-1] != lab[:, 1:]; b[:, :-1] |= d; b[:, 1:] |= d
    D = cv2.distanceTransform((~b).astype(np.uint8), cv2.DIST_L2, 3)
    W = np.ones(lab.shape, dtype=np.float32)
    for c in range(6):
        m = (idx == c).astype(np.uint8)
        if m.sum() == 0:
            continue
        ncc, cc = cv2.connectedComponents(m, connectivity=8)
        if ncc <= 1:
            continue
        means = np.asarray(ndi.mean(D, labels=cc, index=np.arange(1, ncc)), dtype=np.float32)
        R = 2.0 * means                                        # local half-width of the component
        band = np.maximum(np.minimum(taus[c] * R, dmax_px), 1.0)
        bmap = np.zeros(ncc, dtype=np.float32); bmap[1:] = band
        sel = m.astype(bool)
        W[sel] = np.clip(D[sel] / bmap[cc[sel]], wmin, 1.0)
    W[idx == 255] = 1.0
    return W

CACHE = Path(cfg.PREBUILT_CACHE) if cfg.PREBUILT_CACHE else Path(cfg.CACHE_DIR)
MARKER = CACHE / "_done.json"
S = cfg.IMG_SIZE

if MARKER.exists():
    print("cache present, skipping repack ->", CACHE)
else:
    assert not cfg.PREBUILT_CACHE, f"PREBUILT_CACHE set but {MARKER} missing"
    CACHE.mkdir(parents=True, exist_ok=True)
    n = len(records)
    ai = np.lib.format.open_memmap(CACHE / "images.npy", mode="w+", dtype=np.uint8, shape=(n, S, S, 3))
    am = np.lib.format.open_memmap(CACHE / "masks.npy",  mode="w+", dtype=np.uint8, shape=(n, S, S))
    aw = np.lib.format.open_memmap(CACHE / "relw.npy",   mode="w+", dtype=np.uint8, shape=(n, S, S))
    counts = np.zeros((n, 6), dtype=np.int64)
    PX_PER_M = NATIVE_SHAPE[1] / cfg.TILE_METRES
    DMAX_PX = cfg.CSA_DMAX_M * PX_PER_M
    print(f"native {NATIVE_SHAPE[1]}x{NATIVE_SHAPE[0]} px for {cfg.TILE_METRES} m "
          f"-> {PX_PER_M:.2f} px/m | band cap {cfg.CSA_DMAX_M} m = {DMAX_PX:.1f} native px")
    t0 = time.time()
    for i, r in enumerate(tqdm(records, desc="repack")):
        im = cv2.imread(r["img"], cv2.IMREAD_COLOR)
        mk = cv2.imread(r["msk"], cv2.IMREAD_COLOR)
        assert im is not None and mk is not None, r["stem"]
        idx_full = bgr_to_index(mk)
        W_full = csa_weight(idx_full, cfg.CSA_TAU, cfg.CSA_WMIN, DMAX_PX)
        ai[i] = cv2.resize(im, (S, S), interpolation=cv2.INTER_AREA)
        am[i] = cv2.resize(idx_full, (S, S), interpolation=cv2.INTER_NEAREST)
        aw[i] = np.round(np.clip(cv2.resize(W_full, (S, S), interpolation=cv2.INTER_AREA), 0, 1) * 255).astype(np.uint8)
        counts[i] = np.bincount(am[i].ravel(), minlength=256)[:6]
    ai.flush(); am.flush(); aw.flush()
    np.save(CACHE / "counts.npy", counts)
    json.dump({"stems": STEMS, "tiles": TILE_OF_REAL.tolist(), "size": S,
               "native_w": int(NATIVE_SHAPE[1]),
               "csa_tau": list(cfg.CSA_TAU), "csa_wmin": cfg.CSA_WMIN, "csa_dmax_m": cfg.CSA_DMAX_M,
               "tile_metres": cfg.TILE_METRES,
               "minutes": round((time.time() - t0) / 60, 1)}, open(MARKER, "w"))
    print(f"repack done in {(time.time()-t0)/60:.1f} min ->", CACHE)

meta = json.load(open(MARKER))
assert meta["stems"] == STEMS, "cache was built from a different file ordering; delete it and rebuild"
assert meta["size"] == S, f'cache is {meta["size"]} px, cfg wants {S}'
# a part-1 cache predates "native_w"; fall back to the probe decode from section 5
NATIVE_W = int(meta.get("native_w", NATIVE_SHAPE[1]))
COUNTS_REAL = np.load(CACHE / "counts.npy")
print("cache size on disk:", round(sum(f.stat().st_size for f in CACHE.iterdir()) / 1e6), "MB",
      "| native width", NATIVE_W)

## 7. Load the real cache into RAM

Images 963 MB + masks 321 MB + weights 321 MB = ~1.6 GB. Section 9 adds ~1.05 GB of synthetic on
top, which still leaves plenty of the ~13 GB. Preloading removes disk I/O from the training loop
entirely.

In [ ]:
if cfg.PRELOAD_RAM:
    t0 = time.time()
    IMAGES = np.load(CACHE / "images.npy")
    MASKS  = np.load(CACHE / "masks.npy")
    RELW   = np.load(CACHE / "relw.npy")
    print(f"preloaded to RAM: {(IMAGES.nbytes + MASKS.nbytes + RELW.nbytes)/1e9:.2f} GB "
          f"in {time.time()-t0:.0f}s")
else:
    IMAGES = np.load(CACHE / "images.npy", mmap_mode="r")
    MASKS  = np.load(CACHE / "masks.npy",  mmap_mode="r")
    RELW   = np.load(CACHE / "relw.npy",   mmap_mode="r")
    print("memmapped (not preloaded)")

u = np.unique(np.concatenate([np.asarray(MASKS[j])[::32, ::32].ravel()
                              for j in range(0, N_REAL, 7)]))
assert set(u.tolist()) <= set(range(6)) | {255}, f"unexpected label values: {u}"
print("label value check PASSED:", u.tolist())

## 8. Split, and the four-way agreement with the generators

This is **not** part 1's `split_block`. Part 1 dropped the 2-tile buffer around every chunk edge
(260 train tiles at fold 1); Stages 1, 2 and 3 dropped it only around the val block (268). The
synthetic pairs in `fake_pairs_R1` were generated from the 268-tile set, and the B0 and UNetFormer
Stage-4 notebooks use it too.

The sha1 over the sorted train ids is **asserted**, not printed for eyeballing. If S4's real half
were a different split, some synthetic pairs would have been generated from tiles that are now in
validation — the one leakage failure this experiment cannot survive, and it would be invisible in
every metric.

In [ ]:
def block_split(base_ids: List[int], fold: int, n_folds: int, buffer: int):
    n = len(base_ids)
    edges = [round(i * n / n_folds) for i in range(n_folds + 1)]
    lo, hi = edges[fold], edges[fold + 1]
    val = base_ids[lo:hi]
    train = base_ids[:max(0, lo - buffer)] + base_ids[min(n, hi + buffer):]
    return train, val

tr_tiles, va_tiles = block_split(BASE_TILES, cfg.FOLD, cfg.N_FOLDS, cfg.BLOCK_BUFFER)
assert not (set(tr_tiles) & set(va_tiles)), "train/val base-tile overlap"

split_sha1 = hashlib.sha1(",".join(map(str, sorted(tr_tiles))).encode()).hexdigest()
assert split_sha1 == cfg.UPSTREAM_TRAIN_SHA1, (
    "SPLIT DISAGREEMENT with the generators -- the synthetic pairs were made from a different "
    f"train set.\n  upstream: {cfg.UPSTREAM_TRAIN_SHA1}\n  here    : {split_sha1}")

tile_to_rows = {}
for i, t in enumerate(TILE_OF_REAL):
    tile_to_rows.setdefault(int(t), []).append(i)
REAL_TRAIN_IDX = np.array(sorted(j for t in tr_tiles for j in tile_to_rows[t]))
VAL_IDX        = np.array(sorted(j for t in va_tiles for j in tile_to_rows[t]))
assert not (set(TILE_OF_REAL[REAL_TRAIN_IDX].tolist()) & set(TILE_OF_REAL[VAL_IDX].tolist())), "group leakage!"
assert len(REAL_TRAIN_IDX) == 804, len(REAL_TRAIN_IDX)
assert len(VAL_IDX) == 408, len(VAL_IDX)

print(f"{cfg.SPLIT_MODE} fold {cfg.FOLD}: {len(tr_tiles)} train tiles / {len(va_tiles)} val tiles")
print(f"  -> {len(REAL_TRAIN_IDX)} real train pairs / {len(VAL_IDX)} val images")
print(f"  buffer dropped {len(BASE_TILES)-len(tr_tiles)-len(va_tiles)} tiles")
print(f"  train_sha1 {split_sha1[:12]} -- four-way agreement with S1/S2/S3 ASSERTED")

## 9. The synthetic half

Loaded only when `ARM == "R1"`. The Stage-3 contract is re-asserted rather than trusted: 804 pairs,
indices 0..803 complete, images 512x512x3 `uint8` BGR, masks 512x512 `uint8` with values in
{0..5, 255}.

**CSA on synthetic masks is computed at 512, not at native resolution**, because that is the only
resolution a generated mask exists at. The width-normalised term is scale-equivariant — `tau * 2 *
mean(D)` is a fraction of local object width and carries no absolute units — so it transfers
unchanged. The `DMAX` cap does not: it is an absolute distance, and at 512 the equivalent of the
real pipeline's 46.9 native px is `CSA_DMAX_M * IMG_SIZE / TILE_METRES = 10.24` px. The one
remaining difference is that the real field was area-averaged down from native resolution, which
smooths the ramp slightly, while the synthetic field is computed directly on the 512 grid. The mean
weight per class is printed for both below so the size of that difference is visible rather than
assumed.

Synthetic rows live at absolute indices `N_REAL..N_REAL+803` and carry `tile = -(i+1)`. Negative ids
can never collide with a real tile, which makes the assert at the end of this section — that no
synthetic row reached validation — check something real rather than restate an assumption.

In [ ]:
def find_fake_root(explicit, name="fake_pairs_R1"):
    if explicit and Path(explicit).is_dir(): return explicit
    for p in Path("/kaggle/input").rglob(name):
        if p.is_dir() and any(p.glob("genimage*.png")): return str(p)
    raise FileNotFoundError(f"{name} not found -- attach sugam24/geonusaf-fake-pairs-block-fold1")

DMAX_PX_512 = cfg.CSA_DMAX_M * cfg.IMG_SIZE / cfg.TILE_METRES
N_FAKE_LOADED = 0
FAKE_IMAGES = np.zeros((0, S, S, 3), np.uint8)
FAKE_MASKS  = np.zeros((0, S, S), np.uint8)
FAKE_RELW   = np.zeros((0, S, S), np.uint8)
COUNTS_FAKE = np.zeros((0, 6), np.int64)
FAKE_ROOT = None

if USE_FAKE:
    FAKE_ROOT = Path(find_fake_root(cfg.FAKE_ROOT))
    print("FAKE_ROOT:", FAKE_ROOT)
    print(f"CSA band cap at 512: {DMAX_PX_512:.2f} px "
          f"(= real pipeline's {cfg.CSA_DMAX_M * NATIVE_W / cfg.TILE_METRES:.1f} native px)")

    idxs = sorted(int(p.stem[len("genimage"):]) for p in FAKE_ROOT.glob("genimage*.png"))
    assert len(idxs) == cfg.N_FAKE, f"{len(idxs)} synthetic images, expected {cfg.N_FAKE}"
    assert idxs == list(range(cfg.N_FAKE)), "synthetic index range is not 0..n-1"

    FAKE_IMAGES = np.zeros((cfg.N_FAKE, S, S, 3), np.uint8)
    FAKE_MASKS  = np.zeros((cfg.N_FAKE, S, S), np.uint8)
    FAKE_RELW   = np.zeros((cfg.N_FAKE, S, S), np.uint8)
    COUNTS_FAKE = np.zeros((cfg.N_FAKE, 6), np.int64)

    t0 = time.time()
    for i in tqdm(idxs, desc="synthetic + CSA"):
        img = cv2.imread(str(FAKE_ROOT / f"genimage{i}.png"), cv2.IMREAD_COLOR)
        lab = cv2.imread(str(FAKE_ROOT / f"genmask{i}.png"), cv2.IMREAD_GRAYSCALE)
        assert img is not None and lab is not None, f"unreadable synthetic pair {i}"
        assert img.shape == (S, S, 3) and img.dtype == np.uint8, f"synthetic image {i} is {img.shape}"
        assert lab.shape == (S, S) and lab.dtype == np.uint8, f"synthetic mask {i} is {lab.shape}"
        vals = np.unique(lab)
        assert set(vals.tolist()) <= set(range(6)) | {255}, f"synthetic mask {i} values {vals}"
        W = csa_weight(lab, cfg.CSA_TAU, cfg.CSA_WMIN, DMAX_PX_512)
        FAKE_IMAGES[i] = img
        FAKE_MASKS[i] = lab
        FAKE_RELW[i] = np.round(np.clip(W, 0, 1) * 255).astype(np.uint8)
        COUNTS_FAKE[i] = np.bincount(lab.ravel(), minlength=256)[:6]
    N_FAKE_LOADED = cfg.N_FAKE
    print(f"loaded {N_FAKE_LOADED} synthetic pairs (+CSA) in {time.time()-t0:.0f}s, "
          f"{(FAKE_IMAGES.nbytes + FAKE_MASKS.nbytes + FAKE_RELW.nbytes)/1e9:.2f} GB")

# ---- one index space: 0..N_REAL-1 real, N_REAL..N_REAL+N_FAKE_LOADED-1 synthetic ----
COUNTS   = np.concatenate([COUNTS_REAL, COUNTS_FAKE], axis=0)
TILE_OF  = np.concatenate([TILE_OF_REAL,
                           -(np.arange(N_FAKE_LOADED, dtype=np.int32) + 1)])
FAKE_IDX = np.arange(N_REAL, N_REAL + N_FAKE_LOADED)
TRAIN_IDX = np.concatenate([REAL_TRAIN_IDX, FAKE_IDX]).astype(np.int64)
N_ALL = N_REAL + N_FAKE_LOADED

assert not (set(TRAIN_IDX.tolist()) & set(VAL_IDX.tolist())), "row leakage between train and val"
assert all(TILE_OF[j] > 0 for j in VAL_IDX), "SYNTHETIC ROW IN VALIDATION -- stop"

if USE_FAKE:
    real_px = COUNTS_REAL[REAL_TRAIN_IDX].sum(0).astype(np.float64)
    fake_px = COUNTS_FAKE.sum(0).astype(np.float64)
    print(f"\n{'class':<14}{'real train %':>14}{'synthetic %':>13}{'ratio':>8}")
    for i, nm in enumerate(CLASS_NAMES):
        r_, f_ = real_px[i]/real_px.sum()*100, fake_px[i]/fake_px.sum()*100
        print(f"{nm:<14}{r_:13.3f}%{f_:12.3f}%{(f_/max(r_,1e-9)):8.2f}")
    pr = (COUNTS_REAL[REAL_TRAIN_IDX] > 0).mean(0) * 100
    pf = (COUNTS_FAKE > 0).mean(0) * 100
    print(f"\n{'class':<14}{'real presence':>14}{'synth presence':>16}")
    for i, nm in enumerate(CLASS_NAMES):
        print(f"{nm:<14}{pr[i]:13.1f}%{pf[i]:15.1f}%")

    # CSA sanity: the synthetic field must not be systematically harsher than the real one
    def _mean_w_per_class(rows, get):
        wsum = np.zeros(6); wn = np.zeros(6)
        for j in rows:
            m, w = get(j)
            w = w.astype(np.float32) / 255.0
            for c in range(6):
                sel = m == c
                if sel.any(): wsum[c] += w[sel].sum(); wn[c] += sel.sum()
        return np.where(wn > 0, wsum / np.maximum(wn, 1), np.nan)

    sub_r = REAL_TRAIN_IDX[np.linspace(0, len(REAL_TRAIN_IDX) - 1, 40).astype(int)]
    sub_f = np.linspace(0, N_FAKE_LOADED - 1, 40).astype(int)
    mw_r = _mean_w_per_class(sub_r, lambda j: (np.asarray(MASKS[j]), np.asarray(RELW[j])))
    mw_f = _mean_w_per_class(sub_f, lambda j: (FAKE_MASKS[j], FAKE_RELW[j]))
    print(f"\n{'class':<14}{'CSA w real':>12}{'CSA w synth':>13}")
    for i, nm in enumerate(CLASS_NAMES):
        print(f"{nm:<14}{mw_r[i]:12.3f}{mw_f[i]:13.3f}")
    if np.nanmean(mw_f) < 0.80 * np.nanmean(mw_r):
        print("!! synthetic CSA is much harsher than real -- generated polygons are more "
              "fragmented, so more pixels sit near a boundary. Consider CSA_ON_FAKE=False.")

print(f"\narm {cfg.ARM}: {len(TRAIN_IDX)} train rows "
      f"({len(REAL_TRAIN_IDX)} real + {N_FAKE_LOADED} synthetic) | {len(VAL_IDX)} val images")

## 10. Class weights and the sampler frequency table

Both are computed from the **real** train rows only, so `CLASS_W` and the per-record sampler rule
are bit-identical in both arms and the loss is not a second moving variable. The pool composition —
804 rows vs 1608 — is then the only thing that differs, which is exactly the variable under test.
`WEIGHTS_FROM="mixed"` / `SAMPLER_FREQ_FROM="mixed"` recompute over real+synthetic instead; both are
printed either way so the paper can state what the choice cost.

Cui et al. apply `E_c = (1 - beta^n)/(1 - beta)` to per-class **image** counts. Applied to **pixel**
counts (1e6-1e9), `beta^n` underflows to 0 for every class, `E_c` collapses to the constant
`1/(1-beta)` and the weights come out exactly uniform — silently unweighted CE. `EN_BETA_AUTO` sets
`beta = 1 - 1/N`, which recovers a real spread. Note what that reduces to at pixel scale: `E_c`
approaches `n_c`, effective-number weighting becomes ordinary inverse frequency, and **the cap at 10
is what does the stabilising work**. The saturation ratio is printed so this never goes unnoticed.

In [ ]:
def effective_number_weights(counts_per_class, beta_auto, cap):
    n = counts_per_class.astype(np.float64)
    N = n.sum()
    beta = 1.0 - 1.0 / N if beta_auto else 0.9999
    with np.errstate(over="ignore"):
        eff = (1.0 - np.power(beta, n)) / (1.0 - beta)
    ratio = eff.max() / max(eff.min(), 1e-12)
    w = 1.0 / np.maximum(eff, 1e-12)
    w = w / w.mean()
    w = np.minimum(w, cap)
    w = w / w.mean()
    return w.astype(np.float32), beta, ratio

px_real  = COUNTS[REAL_TRAIN_IDX].sum(0)
px_mixed = COUNTS[TRAIN_IDX].sum(0)
W_REAL,  BETA_R, RATIO_R = effective_number_weights(px_real,  cfg.EN_BETA_AUTO, cfg.W_CAP)
W_MIXED, BETA_M, RATIO_M = effective_number_weights(px_mixed, cfg.EN_BETA_AUTO, cfg.W_CAP)
CLASS_W = W_REAL if cfg.WEIGHTS_FROM == "real" else W_MIXED
BETA, EFF_RATIO = (BETA_R, RATIO_R) if cfg.WEIGHTS_FROM == "real" else (BETA_M, RATIO_M)

if EFF_RATIO < 1.05:
    print("!! SATURATION WARNING: effective numbers are near-constant -> weights effectively uniform.")
print(f"WEIGHTS_FROM = {cfg.WEIGHTS_FROM} | beta = {BETA:.12f} | spread = {EFF_RATIO:,.1f}x\n")
print(f"{'class':<14}{'real px %':>11}{'w(real)':>10}{'w(mixed)':>11}")
for c, nm in enumerate(CLASS_NAMES):
    print(f"{nm:<14}{px_real[c]/px_real.sum()*100:10.2f}%{W_REAL[c]:10.3f}{W_MIXED[c]:11.3f}")
print(f"weight ratio max/min: real {W_REAL.max()/W_REAL.min():.1f} | "
      f"mixed {W_MIXED.max()/W_MIXED.min():.1f}")
CLASS_W_T = torch.tensor(CLASS_W, device=DEVICE)

# tile-presence frequency drives the balanced sampler (section 11)
PRESENCE = (COUNTS > 0)
FREQ_REAL  = PRESENCE[REAL_TRAIN_IDX].mean(0)
FREQ_MIXED = PRESENCE[TRAIN_IDX].mean(0)
SAMPLER_FREQ = FREQ_REAL if cfg.SAMPLER_FREQ_FROM == "real" else FREQ_MIXED
print(f"\nSAMPLER_FREQ_FROM = {cfg.SAMPLER_FREQ_FROM}")
print(f"{'class':<14}{'presence real':>14}{'presence mixed':>16}")
for c, nm in enumerate(CLASS_NAMES):
    print(f"{nm:<14}{FREQ_REAL[c]*100:13.1f}%{FREQ_MIXED[c]*100:15.1f}%")

## 11. Dataset, balanced sampler, DataLoaders — with the augmentation bug fixed

Part 1 seeded a fresh `RandomState` from `self.epoch` and relied on the main process updating
`train_ds.epoch`. With `persistent_workers=True` that update never crossed the process boundary, so
augmentation was frozen after the first pass. Here the draws come from the worker-local `np.random`
stream, which `worker_init_fn` seeds once per worker and which then advances on every call — no
epoch bookkeeping to propagate, nothing to go stale. The assert below draws the same row twice and
requires the results to differ.

**No hflip/vflip** — they are baked into the 1224 real files. On-the-fly augmentation is `rot90`
with `k in {0,1,2,3}` (exact, no resampling, so 10-20 px roads stay crisp; with the baked-in flips
this completes the D4 dihedral group) plus +/-10% brightness and contrast. **No scale jitter** — it
changes effective GSD and would muddy the thin-structure result. The reliability weight map is
rotated with the image and mask.

**The sampler.** Lovasz in `present` mode only accumulates gradient for classes appearing in the
batch, so River — in a small fraction of tiles — gets sparse, spiky signal. Each row is weighted by
the inverse tile-presence frequency of the rarest class it contains, capped at `SAMPLER_CAP`, drawn
with replacement. In R1 the synthetic rows enter the same draw: they carry River in roughly twice as
many masks as the real rows do, so the sampler's rare-class pool is genuinely larger rather than the
same tiles being drawn more often. That is the mechanism this whole experiment is testing, and it is
worth saying so plainly in the paper.

In [ ]:
IMNET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMNET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# absolute row -> (image u8 BGR, mask u8, csa weight u8); real and synthetic look identical
def read_row(j):
    if j < N_REAL:
        return np.asarray(IMAGES[j]), np.asarray(MASKS[j]), np.asarray(RELW[j])
    k = j - N_REAL
    return FAKE_IMAGES[k], FAKE_MASKS[k], FAKE_RELW[k]

class GeoDataset(Dataset):
    def __init__(self, rows, train, cfg):
        self.rows, self.train, self.cfg = np.asarray(rows), train, cfg
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        j = int(self.rows[i])
        im_u8, mk_u8, w_u8 = read_row(j)
        img = im_u8[:, :, ::-1].astype(np.float32) / 255.0          # BGR store -> RGB
        msk = mk_u8.astype(np.int64)
        wgt = w_u8.astype(np.float32) / 255.0
        is_fake = j >= N_REAL
        if (not self.cfg.CSA_ENABLE) or (is_fake and not self.cfg.CSA_ON_FAKE):
            wgt = np.ones_like(wgt)
        if self.train:
            if self.cfg.AUG_FLIP_FAKE and is_fake:
                # real rows carry hf/vf as separate files; this gives synthetic ones the same
                # flip coverage instead of leaving them with only rot90
                f = int(np.random.randint(3))
                if f == 1:   img, msk, wgt = img[:, ::-1], msk[:, ::-1], wgt[:, ::-1]
                elif f == 2: img, msk, wgt = img[::-1], msk[::-1], wgt[::-1]
            if self.cfg.AUG_ROT90:
                k = int(np.random.randint(4))
                if k:
                    img = np.rot90(img, k, (0, 1)); msk = np.rot90(msk, k); wgt = np.rot90(wgt, k)
            if self.cfg.AUG_PHOTOMETRIC:
                b = np.random.uniform(-self.cfg.AUG_BRIGHTNESS, self.cfg.AUG_BRIGHTNESS)
                c = 1.0 + np.random.uniform(-self.cfg.AUG_CONTRAST, self.cfg.AUG_CONTRAST)
                img = np.clip((img - 0.5) * c + 0.5 + b, 0.0, 1.0)
        img = (img - IMNET_MEAN) / IMNET_STD
        return (torch.from_numpy(np.ascontiguousarray(img.transpose(2, 0, 1))),
                torch.from_numpy(np.ascontiguousarray(msk)),
                torch.from_numpy(np.ascontiguousarray(wgt)))

def make_sampler(rows, presence, tile_freq, cap, generator=None):
    inv = 1.0 / np.maximum(tile_freq, 1e-6)
    w = np.array([max(inv[c] for c in range(6) if presence[j, c]) if presence[j].any() else 1.0
                  for j in rows], dtype=np.float64)
    w = np.minimum(w / w.min(), cap)
    return WeightedRandomSampler(torch.as_tensor(w), num_samples=len(rows),
                                 replacement=True, generator=generator), w

train_ds = GeoDataset(TRAIN_IDX, True, cfg)
val_ds   = GeoDataset(VAL_IDX,  False, cfg)

sampler, SAMPLER_W = (make_sampler(TRAIN_IDX, PRESENCE, SAMPLER_FREQ, cfg.SAMPLER_CAP, g)
                      if cfg.BALANCED_SAMPLER else (None, None))
if sampler is not None and N_FAKE_LOADED:
    wr = SAMPLER_W[:len(REAL_TRAIN_IDX)]; wf = SAMPLER_W[len(REAL_TRAIN_IDX):]
    print(f"sampler weight mean: real {wr.mean():.3f} | synthetic {wf.mean():.3f} "
          f"| synthetic share of total draw mass {wf.sum()/SAMPLER_W.sum()*100:.1f}%")

# the bug check: two draws of the same row must differ
np.random.seed(0)
_a, _, _ = train_ds[0]
_b, _, _ = train_ds[0]
assert not torch.allclose(_a, _b), (
    "augmentation is STATIC -- two draws of row 0 are identical. This is the persistent_workers "
    "bug from the part-1 notebook and it must not ship.")
print("augmentation liveness check PASSED (two draws of row 0 differ)")

_VAL_WORKERS = min(2, cfg.NUM_WORKERS)
train_dl = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, sampler=sampler,
                      shuffle=(sampler is None), drop_last=True,
                      num_workers=cfg.NUM_WORKERS, pin_memory=True,
                      persistent_workers=cfg.NUM_WORKERS > 0,
                      worker_init_fn=worker_init_fn)
val_dl   = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                      num_workers=_VAL_WORKERS, pin_memory=True,
                      persistent_workers=_VAL_WORKERS > 0,
                      worker_init_fn=worker_init_fn)

steps_per_pass = len(train_dl)
xb, yb, wb = next(iter(train_dl))
print("batch:", tuple(xb.shape), xb.dtype, "| labels", sorted(set(yb.unique().tolist()))[:8],
      "| w range", round(float(wb.min()), 3), round(float(wb.max()), 3))
assert set(yb.unique().tolist()).issubset(set(range(6)) | {255})
print(f"steps per pass: {steps_per_pass} | {cfg.MAX_STEPS} steps = "
      f"{cfg.MAX_STEPS/steps_per_pass:.1f} passes over the training pool")

## 12. Loss — weighted CE x CSA + Lovasz (CSA-masked, step-ramped) + soft-clDice

Unchanged from part 1 except that `set_epoch` became `set_step`. Three design decisions carry over:

**1. The whole loss is fp32 outside autocast.** Lovasz sorts per-pixel errors and its gradient is a
cumsum difference, which accumulates visible error and can NaN in fp16. `ignore_index` is filtered
before flattening so 255 never reaches either term.

**2. CSA enters CE as a weight but Lovasz as a hard mask.** Lovasz is a set-function surrogate over
the *sorted* error vector — there is no principled way to scale individual pixels inside it. Worse,
it concentrates gradient exactly where errors cluster, i.e. at boundaries, so a weighted CE and an
unweighted Lovasz would actively fight. Pixels with `w < CSA_LOVASZ_THRESH` are folded into the
ignore set for the Lovasz term only.

**3. Lovasz ramps from 0 over the first 500 steps.** Its gradient is defined by the error *ranking*;
before the network can rank anything it emits large near-random updates — exactly when the freshly
initialised detail-path gate is deciding whether to open.

**soft-clDice** (Shit et al., CVPR 2021) is a topology-aware auxiliary on Road and River only, via
iterative soft erosion / opening, computed at 256 and skipped per-sample when the class is absent
from the ground truth.

The printed **total is non-stationary by construction** — `lambda` ramps and `mu` switches on at
step 1000 — so only `train_ce` / `val_ce` are comparable across the run.

In [ ]:
def lovasz_grad(gt_sorted):
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    inter = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jacc = 1.0 - inter / union
    if p > 1:
        jacc[1:p] = jacc[1:p] - jacc[0:-1]
    return jacc

def lovasz_softmax_flat(probs, labels, classes="present"):
    if probs.numel() == 0:
        return probs.sum() * 0.0
    C = probs.size(1)
    losses = []
    for c in range(C):
        fg = (labels == c).float()
        if classes == "present" and fg.sum() == 0:
            continue
        errors = (fg - probs[:, c]).abs()
        errors_sorted, perm = torch.sort(errors, 0, descending=True)
        losses.append(torch.dot(errors_sorted, lovasz_grad(fg[perm])))
    if not losses:
        return probs.sum() * 0.0
    return torch.stack(losses).mean()

def soft_erode(x):
    p1 = -F.max_pool2d(-x, (3, 1), (1, 1), (1, 0))
    p2 = -F.max_pool2d(-x, (1, 3), (1, 1), (0, 1))
    return torch.min(p1, p2)

def soft_open(x):
    return F.max_pool2d(soft_erode(x), (3, 3), (1, 1), (1, 1))

def soft_skel(x, iters):
    x1 = soft_open(x); skel = F.relu(x - x1)
    for _ in range(iters):
        x = soft_erode(x); x1 = soft_open(x)
        delta = F.relu(x - x1)
        skel = skel + F.relu(delta - skel * delta)
    return skel

def soft_cldice(prob_c, gt_c, iters, eps=1.0):
    sp, sg = soft_skel(prob_c, iters), soft_skel(gt_c, iters)
    tprec = (torch.sum(sp * gt_c, (1, 2, 3)) + eps) / (torch.sum(sp, (1, 2, 3)) + eps)
    tsens = (torch.sum(sg * prob_c, (1, 2, 3)) + eps) / (torch.sum(sg, (1, 2, 3)) + eps)
    return 1.0 - 2.0 * tprec * tsens / (tprec + tsens)

class GeoLoss(nn.Module):
    def __init__(self, cfg, class_w):
        super().__init__(); self.cfg = cfg; self.register_buffer("cw", class_w)
        self.lam = 0.0; self.mu = 0.0
    def set_step(self, step):
        c = self.cfg
        self.lam = c.LOVASZ_LAMBDA * min(1.0, (step + 1) / max(c.LOVASZ_WARMUP_STEPS, 1))
        self.mu = c.CLDICE_MU if (c.CLDICE_ENABLE and step >= c.CLDICE_START_STEP) else 0.0
    def forward(self, logits, target, relw):
        c = self.cfg
        logits = logits.float()                                   # fp32, outside autocast
        valid = target != c.IGNORE_INDEX
        ce_map = F.cross_entropy(logits, target.clamp(max=c.NUM_CLASSES - 1), weight=self.cw,
                                 reduction="none", label_smoothing=c.LABEL_SMOOTH)
        ce = (ce_map * relw * valid).sum() / (relw * valid).sum().clamp(min=1.0)
        probs = logits.softmax(1)
        keep = valid & (relw >= c.CSA_LOVASZ_THRESH)              # hard mask, not a weight
        if keep.sum() > 0 and self.lam > 0:
            pf = probs.permute(0, 2, 3, 1)[keep]
            lv = lovasz_softmax_flat(pf, target[keep])
        else:
            lv = logits.sum() * 0.0
        cl = logits.sum() * 0.0
        if self.mu > 0:
            r = c.CLDICE_RES
            pr = F.adaptive_avg_pool2d(probs, r)
            terms = []
            for ci in c.CLDICE_CLASSES:
                gt = ((target == ci) & valid).float().unsqueeze(1)
                gt = F.adaptive_avg_pool2d(gt, r)
                present = (gt.sum((1, 2, 3)) > 0)
                if present.any():
                    terms.append(soft_cldice(pr[:, ci:ci + 1][present], gt[present], c.CLDICE_ITERS).mean())
            if terms:
                cl = torch.stack(terms).mean()
        total = ce + self.lam * lv + self.mu * cl
        return total, dict(ce=float(ce), lovasz=float(lv), cldice=float(cl), lam=self.lam, mu=self.mu)

criterion = GeoLoss(cfg, CLASS_W_T).to(DEVICE)
print("loss ready | lovasz lambda target", cfg.LOVASZ_LAMBDA, "| clDice mu", cfg.CLDICE_MU)

## 13. Model, optimizer, step-based scheduler, EMA

TC-SegFormer verbatim from part 1: MiT-B0 encoder + the pretrained all-MLP decode head, with a
residual H/2 tail.

```
image 512 --> MiT encoder --> 4 stages --> linear_c + linear_fuse --> fused (D, 128, 128)
   |                                                                     |
   |                                                        1x1 proj to 64 ch
   |                                                                     |
   +--> detail path (3->8->16->32, stride 2, GroupNorm) --> (32, 256, 256)
                                        |                                |
                                        +---- gate * detail --concat-----+
                                                       |
                                              3x3 fuse -> 64 -> 1x1 refine -> residual @ H/2 -> x2
```

*The refinement is a residual, not a replacement.* The base path goes straight through
`head.classifier` exactly as in plain SegFormer; the H/2 branch adds a correction whose final conv
is initialised at std 1e-4, so at step 0 the branch is very nearly inert and `USE_DETAIL=False`
reproduces plain SegFormer-B0 **bit-exactly**.

Part 1 asserted the inertness in prose ("vanilla to four decimal places") and never checked it. The
cell below **measures** it on the real pretrained backbone and prints the residual, the logit scale,
and the percentage of pixels whose predicted class the residual flips — the last of those is the
number that actually matters, because the ablation argument is that TC-SegFormer *starts* as
SegFormer-B0. It warns if more than 1% of pixels move.

*Three LR tiers.* Encoder 6e-6, pretrained decode head 6e-5, randomly-initialised modules
(`proj`/`detail`/`fuse`/`refine`/`gate`) 6e-4. Norm and bias parameters are excluded from decay.

*GroupNorm, not BatchNorm,* in the new modules — SegFormer is LayerNorm throughout, and adding BN at
small batch under EMA invites a train/eval statistics discrepancy the rest of the network does not
have.

The only change from part 1 is the scheduler horizon: `TOTAL_STEPS` is `COS_HORIZON_STEPS` straight
from the config rather than `steps_per_epoch * MAX_EPOCHS`. That single line is what makes the two
arms compute-matched — under the old form, R1's doubled `steps_per_pass` would have stretched the
cosine to 12000 steps and given it twice the schedule.

In [ ]:
from transformers import SegformerConfig, SegformerForSemanticSegmentation

def head_mlps(head):
    # transformers <5 calls this ModuleList `linear_c`; 5.x renamed it `linear_projections`.
    # The name is part of every state-dict key, which is why section 1 pins <5 -- this shim only
    # keeps the forward pass from dying with an AttributeError if the pin is ever bypassed.
    for name in ("linear_c", "linear_projections"):
        if hasattr(head, name): return getattr(head, name), name
    raise AttributeError("decode_head has neither linear_c nor linear_projections")

class DetailPath(nn.Module):
    def __init__(self, chs):
        super().__init__()
        c1, c2, c3 = chs
        def blk(i, o, s): return nn.Sequential(nn.Conv2d(i, o, 3, s, 1, bias=False),
                                               nn.GroupNorm(min(8, o), o), nn.ReLU(inplace=True))
        self.net = nn.Sequential(blk(3, c1, 2), blk(c1, c2, 1), blk(c2, c3, 1))
        self.out_ch = c3
    def forward(self, x): return self.net(x)

class TCSegFormer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        conf = SegformerConfig.from_pretrained(cfg.BACKBONE)
        conf.num_labels = cfg.NUM_CLASSES
        conf.drop_path_rate = cfg.DROP_PATH
        base = SegformerForSemanticSegmentation.from_pretrained(
            cfg.BACKBONE, config=conf, ignore_mismatched_sizes=True)
        self.encoder = base.segformer
        self.head = base.decode_head
        D = conf.decoder_hidden_size
        self.use_detail = cfg.USE_DETAIL
        if self.use_detail:
            self.proj = nn.Conv2d(D, cfg.FUSE_CH, 1)
            self.detail = DetailPath(cfg.DETAIL_CH)
            self.gate = nn.Parameter(torch.tensor(float(cfg.DETAIL_GATE_INIT)))
            self.fuse = nn.Sequential(
                nn.Conv2d(cfg.FUSE_CH + self.detail.out_ch, cfg.FUSE_CH, 3, 1, 1, bias=False),
                nn.GroupNorm(8, cfg.FUSE_CH), nn.ReLU(inplace=True))
            self.refine = nn.Conv2d(cfg.FUSE_CH, cfg.NUM_CLASSES, 1)
            # near-zero init: at step 0 the refinement contributes ~nothing, so the model IS
            # vanilla SegFormer. Small-normal rather than exactly zero so proj/fuse/detail
            # receive gradient from the very first step instead of waiting one update.
            nn.init.normal_(self.refine.weight, std=1e-4); nn.init.zeros_(self.refine.bias)

    def fused_feature(self, hidden_states):
        bs = hidden_states[-1].shape[0]
        outs = []
        for hs, mlp in zip(hidden_states, head_mlps(self.head)[0]):
            h, w = hs.shape[2], hs.shape[3]
            x = mlp(hs).permute(0, 2, 1).reshape(bs, -1, h, w)
            x = F.interpolate(x, size=hidden_states[0].shape[2:], mode="bilinear", align_corners=False)
            outs.append(x)
        x = self.head.linear_fuse(torch.cat(outs[::-1], dim=1))
        x = self.head.batch_norm(x)
        x = self.head.activation(x)
        return self.head.dropout(x)

    def forward(self, x):
        H, W = x.shape[-2:]
        hs = self.encoder(x, output_hidden_states=True, return_dict=True).hidden_states
        fused = self.fused_feature(hs)                              # (B, D, H/4, W/4)
        base = self.head.classifier(fused)                          # vanilla SegFormer logits @ H/4
        logits = F.interpolate(base, size=(H, W), mode="bilinear", align_corners=False)
        if not self.use_detail:
            return logits                                           # == plain SegFormer, exactly
        f = F.interpolate(self.proj(fused), size=(H // 2, W // 2), mode="bilinear", align_corners=False)
        f = torch.cat([f, self.gate * self.detail(x)], dim=1)
        r = self.refine(self.fuse(f))                               # (B, 6, H/2, W/2) residual
        return logits + F.interpolate(r, size=(H, W), mode="bilinear", align_corners=False)

NEW_PREFIX = ("proj", "detail", "fuse", "refine", "gate")

def build_model():
    return TCSegFormer(cfg).to(DEVICE).to(memory_format=torch.channels_last)

model = build_model()
print("decode-head MLP list resolved as:", head_mlps(model.head)[1],
      "| transformers", transformers.__version__)
n_all = sum(p.numel() for p in model.parameters())
n_new = sum(p.numel() for n, p in model.named_parameters() if n.split(".")[0] in NEW_PREFIX)
ARCH_SIG = hashlib.sha1(
    "|".join(f"{k}:{tuple(v.shape)}" for k, v in sorted(model.state_dict().items())).encode()
).hexdigest()[:12]

def param_groups(model, cfg):
    gp = {k: [] for k in ("enc_d", "enc_n", "head_d", "head_n", "new_d", "new_n")}
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        root = n.split(".")[0]
        tier = "new_" if root in NEW_PREFIX else ("enc_" if root == "encoder" else "head_")
        no_decay = p.ndim <= 1 or "norm" in n.lower() or n.endswith(".bias") or n == "gate"
        gp[tier + ("n" if no_decay else "d")].append(p)
    wd = cfg.WEIGHT_DECAY
    return [dict(params=gp["enc_d"],  lr=cfg.LR_ENCODER, weight_decay=wd),
            dict(params=gp["enc_n"],  lr=cfg.LR_ENCODER, weight_decay=0.0),
            dict(params=gp["head_d"], lr=cfg.LR_HEAD,    weight_decay=wd),
            dict(params=gp["head_n"], lr=cfg.LR_HEAD,    weight_decay=0.0),
            dict(params=gp["new_d"],  lr=cfg.LR_NEW,     weight_decay=wd),
            dict(params=gp["new_n"],  lr=cfg.LR_NEW,     weight_decay=0.0)]

optimizer = torch.optim.AdamW(param_groups(model, cfg))
LR_IDX_ENC, LR_IDX_HEAD, LR_IDX_NEW = 0, 2, 4
for nm, grp in zip(["enc_d", "enc_n", "head_d", "head_n", "new_d", "new_n"], optimizer.param_groups):
    print(f"  {nm:<7} lr {grp['lr']:.1e}  wd {grp['weight_decay']}  tensors {len(grp['params'])}")

scaler = torch.cuda.amp.GradScaler(enabled=cfg.AMP)
TOTAL_STEPS = cfg.COS_HORIZON_STEPS          # steps, NOT epochs -- this is the arm-matching line

def lr_lambda(step):
    if step < cfg.WARMUP_STEPS:
        return (step + 1) / cfg.WARMUP_STEPS
    p = (step - cfg.WARMUP_STEPS) / max(1, TOTAL_STEPS - cfg.WARMUP_STEPS)
    return 0.5 * (1 + math.cos(math.pi * min(p, 1.0)))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float() for k, v in model.state_dict().items()
                       if v.dtype.is_floating_point}
        self.backup = None
    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)
    def apply_to(self, model):
        self.backup = {k: v.detach().clone() for k, v in model.state_dict().items() if k in self.shadow}
        model.load_state_dict({k: (self.shadow[k].to(v.dtype) if k in self.shadow else v)
                               for k, v in model.state_dict().items()}, strict=True)
    def restore(self, model):
        if self.backup is None: return
        sd = model.state_dict(); sd.update(self.backup); model.load_state_dict(sd, strict=True)
        self.backup = None

ema = EMA(model, cfg.EMA_DECAY) if cfg.EMA else None

with torch.no_grad(), torch.cuda.amp.autocast(enabled=cfg.AMP):
    probe_out = model(torch.zeros(2, 3, cfg.IMG_SIZE, cfg.IMG_SIZE,
                                  device=DEVICE).to(memory_format=torch.channels_last))
assert probe_out.shape[-2:] == (cfg.IMG_SIZE, cfg.IMG_SIZE), probe_out.shape

# ---- MEASURE the "at step 0 this IS vanilla SegFormer" property instead of asserting it ----
# Part 1 claimed four decimal places in prose and never checked. On the real pretrained backbone
# the residual is a fixed ~1e-3 in logit units (refine is init'd at std 1e-4 over FUSE_CH inputs),
# so what matters is its size RELATIVE to the logit scale, and how many pixels it can flip.
if cfg.USE_DETAIL:
    model.eval()
    with torch.no_grad():
        _x = torch.randn(2, 3, cfg.IMG_SIZE, cfg.IMG_SIZE, device=DEVICE).to(memory_format=torch.channels_last)
        _full = model(_x)
        _hs = model.encoder(_x, output_hidden_states=True, return_dict=True).hidden_states
        _van = F.interpolate(model.head.classifier(model.fused_feature(_hs)),
                             size=_x.shape[-2:], mode="bilinear", align_corners=False)
    model.train()
    _res = (_full - _van).abs().max().item()
    _scale = _van.abs().mean().item()
    _flip = (_full.argmax(1) != _van.argmax(1)).float().mean().item()
    print(f"step-0 residual vs vanilla SegFormer: max |d| {_res:.2e} | logit scale {_scale:.3f} "
          f"| ratio {_res/max(_scale,1e-12):.2e} | pixels flipped {_flip*100:.3f}%")
    if _flip > 0.01:
        print("!! the detail path is NOT inert at init -- more than 1% of pixels already change "
              "class. Lower the refine init std; the ablation claim depends on this.")
    del _full, _van, _hs, _x; gc.collect(); torch.cuda.empty_cache()
print(f"\nparams total {n_all/1e6:.2f} M | new modules {n_new/1e6:.3f} M "
      f"(+{100*n_new/(n_all-n_new):.2f}%) | arch_sig {ARCH_SIG}")
print("logits after upsample:", tuple(probe_out.shape))
print(f"cosine horizon: {TOTAL_STEPS} steps (identical in both arms) | EMA: {bool(ema)}")

## 14. Metrics

One 6x6 confusion matrix accumulated on-GPU with `bincount`, ignore pixels dropped before
accumulation; IoU, F1, OA, fwIoU, kappa, producer's and user's accuracy all derived from it so
nothing can drift out of sync.

**Per-class precision and recall are reported separately, and this matters.** F1 is algebraically
`2*IoU/(1+IoU)` and carries no additional information. Producer's (recall) and user's (precision)
accuracy are what tell you whether River at 0.14 is under- or over-prediction, and those need
opposite remedies — which is exactly the question the synthetic River pixels are meant to move.

A second matrix is accumulated over the **core region** only (`relw >= CORE_THRESH`), giving the
matched full/core protocol. `ConfMat` is also used per base tile in section 20, which is what the
bootstrap resamples.

In [ ]:
from skimage.morphology import skeletonize

@torch.no_grad()
def update_cm(cm, pred, target, mask=None):
    v = target != cfg.IGNORE_INDEX
    if mask is not None: v = v & mask
    if v.sum() == 0: return cm
    t, p = target[v].reshape(-1), pred[v].reshape(-1)
    cm += torch.bincount(t * cfg.NUM_CLASSES + p,
                         minlength=cfg.NUM_CLASSES ** 2).reshape(cfg.NUM_CLASSES, -1)
    return cm

def new_cm(): return torch.zeros(cfg.NUM_CLASSES, cfg.NUM_CLASSES, dtype=torch.long, device=DEVICE)

def cm_metrics(cm):
    cm = cm.double().cpu().numpy() if torch.is_tensor(cm) else np.asarray(cm, dtype=np.float64)
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    den = tp + fp + fn
    iou = np.where(den > 0, tp / np.maximum(den, 1e-9), np.nan)
    f1 = np.where((2 * tp + fp + fn) > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1e-9), np.nan)
    prec = np.where((tp + fp) > 0, tp / np.maximum(tp + fp, 1e-9), np.nan)     # user's accuracy
    rec = np.where((tp + fn) > 0, tp / np.maximum(tp + fn, 1e-9), np.nan)      # producer's accuracy
    N = cm.sum(); oa = tp.sum() / max(N, 1e-9)
    freq = cm.sum(1) / max(N, 1e-9)
    fwiou = float(np.nansum(freq * iou))
    pe = float((cm.sum(0) * cm.sum(1)).sum() / max(N * N, 1e-9))
    kappa = (oa - pe) / max(1 - pe, 1e-9)
    return dict(mIoU=float(np.nanmean(iou)), mF1=float(np.nanmean(f1)), OA=float(oa),
                fwIoU=fwiou, kappa=float(kappa),
                IoU={n: float(v) for n, v in zip(CLASS_NAMES, iou)},
                F1={n: float(v) for n, v in zip(CLASS_NAMES, f1)},
                UA={n: float(v) for n, v in zip(CLASS_NAMES, prec)},
                PA={n: float(v) for n, v in zip(CLASS_NAMES, rec)},
                support_px={n: int(v) for n, v in zip(CLASS_NAMES, cm.sum(1))},
                cm_raw=cm.astype(np.int64).tolist(),
                cm_norm=(cm / np.maximum(cm.sum(1, keepdims=True), 1e-9)).tolist())

def cldice_pair(pred_b, gt_b, eps=1e-6):
    if pred_b.sum() == 0 and gt_b.sum() == 0: return None
    sp = skeletonize(pred_b) if pred_b.any() else np.zeros_like(pred_b)
    sg = skeletonize(gt_b) if gt_b.any() else np.zeros_like(gt_b)
    tprec = (sp & gt_b).sum() / (sp.sum() + eps) if sp.sum() else 0.0
    tsens = (sg & pred_b).sum() / (sg.sum() + eps) if sg.sum() else 0.0
    return 0.0 if (tprec + tsens) == 0 else float(2 * tprec * tsens / (tprec + tsens))

print("metrics ready")

## 15. Efficiency profile

Both arms run the same architecture, so these are identical by construction — they are recorded so
the Stage-4 table can stand alone rather than pointing back at part 1, and because determinism is
now off these FPS numbers are finally comparable across the three Stage-4 models.

Two T4 facts worth stating in the paper: sm_75 has no bf16 and no FlashAttention-2, so this runs on
fp16 + GradScaler with standard attention. SegFormer's sequence-reduction attention keeps stage 1
tractable, so attention is **not** the scaling bottleneck here; stage-1 MLP activations and the
decoder concat are.

In [ ]:
def efficiency_report(model, cfg):
    rep = {"params_M": sum(p.numel() for p in model.parameters()) / 1e6,
           "params_new_M": sum(p.numel() for n, p in model.named_parameters()
                               if n.split(".")[0] in NEW_PREFIX) / 1e6}
    was_training = model.training
    try:
        from fvcore.nn import FlopCountAnalysis
        model.eval()
        fa = FlopCountAnalysis(model, torch.zeros(1, 3, cfg.IMG_SIZE, cfg.IMG_SIZE, device=DEVICE))
        fa.unsupported_ops_warnings(False); fa.uncalled_modules_warnings(False)
        rep["GFLOPs_512"] = fa.total() / 1e9
    except Exception as e:
        rep["GFLOPs_512"] = None; rep["flops_error"] = str(e)[:120]
    model.eval()
    x = torch.zeros(1, 3, cfg.IMG_SIZE, cfg.IMG_SIZE, device=DEVICE).to(memory_format=torch.channels_last)
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=cfg.AMP):
        for _ in range(20): model(x)
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(50): model(x)
        torch.cuda.synchronize()
    rep["infer_FPS_bs1"] = 50 / (time.time() - t0)
    if was_training: model.train()
    return rep

EFF = efficiency_report(model, cfg)
print(json.dumps(EFF, indent=1))

## 16. Hugging Face persistence and resume

One repo per arm, so `R0` and `R1` never overwrite each other.

* **`best.pt`** — the EMA weights merged into a full state dict, plus config and metrics, pushed on
  every val-mIoU improvement. This is what section 23 downloads.
* **`last.pt`** — full optimizer / scaler / scheduler / EMA state plus the step counter, pushed every
  5 validations, so a killed session resumes mid-run rather than restarting.

**Architecture guard.** A persistent per-arm repo means a checkpoint written by an older version of
the model is still sitting there after any architecture change, and `load_state_dict` would fail on
it. Every checkpoint carries `arch` and `arch_sig`, and resume checks both **before** touching the
model, because a partially-applied `load_state_dict` leaves the network in a mixed state. On
mismatch the run starts fresh and deletes the stale files so later sessions of the *new* run resume
cleanly.

In [ ]:
from huggingface_hub import HfApi, create_repo, hf_hub_download
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN"); print("secret fallback:", e)

api = HfApi(token=HF_TOKEN) if HF_TOKEN else None
if api and cfg.HF_PUSH:
    create_repo(HF_REPO, token=HF_TOKEN, exist_ok=True, private=False)

def hf_push(local, name, msg="update"):
    if not (api and cfg.HF_PUSH): return False
    for attempt in range(3):
        try:
            api.upload_file(path_or_fileobj=str(local), path_in_repo=name,
                            repo_id=HF_REPO, token=HF_TOKEN, commit_message=msg)
            return True
        except Exception as e:
            print(f"    push {name} failed ({type(e).__name__}), retry {attempt+1}/3")
            time.sleep(5)
    return False

def hf_try_download(name, repo=None):
    if not api: return None
    try: return hf_hub_download(repo or HF_REPO, name, token=HF_TOKEN)
    except Exception: return None

def push_model_card(m, step):
    L = ["---",
         "tags: [semantic-segmentation, remote-sensing, segformer, geonusaf, synthetic-data]",
         "---", "",
         f"# GeoNUSAF Stage 4 - TC-SegFormer - arm {cfg.ARM} - {cfg.SPLIT_MODE} fold {cfg.FOLD}", "",
         "Kathmandu Valley land-use segmentation, 6 classes, ignore_index=255, 512x512.", "",
         "| field | value |", "|---|---|",
         f"| arm | {cfg.ARM} ({'real only' if not USE_FAKE else 'real + 804 synthetic'}) |",
         f"| train rows | {len(TRAIN_IDX)} ({len(REAL_TRAIN_IDX)} real + {N_FAKE_LOADED} synthetic) |",
         f"| synthetic source | sugam24/geonusaf-stage3-fakepairs-block-fold1, fake_pairs_R1 |",
         f"| split | {cfg.SPLIT_MODE} fold {cfg.FOLD}, train_sha1 `{split_sha1[:12]}` |",
         f"| backbone | `{cfg.BACKBONE}` |",
         f"| detail path | {cfg.USE_DETAIL} (ch={list(cfg.DETAIL_CH)}, fuse={cfg.FUSE_CH}) |",
         f"| CSA | {cfg.CSA_ENABLE} (tau={list(cfg.CSA_TAU)}, w_min={cfg.CSA_WMIN}, on_fake={cfg.CSA_ON_FAKE}) |",
         f"| soft-clDice | {cfg.CLDICE_ENABLE} (mu={cfg.CLDICE_MU} from step {cfg.CLDICE_START_STEP}) |",
         f"| balanced sampler | {cfg.BALANCED_SAMPLER} (cap {cfg.SAMPLER_CAP}, freq from {cfg.SAMPLER_FREQ_FROM}) |",
         f"| schedule | {cfg.MAX_STEPS} steps, warmup {cfg.WARMUP_STEPS}, cosine {cfg.COS_HORIZON_STEPS} |",
         f"| class weights | from {cfg.WEIGHTS_FROM} |",
         f"| seed | {cfg.SEED} |",
         f"| best step | {step} |",
         f"| val mIoU | {m.get('mIoU', float('nan')):.4f} |",
         f"| val mF1 | {m.get('mF1', float('nan')):.4f} |",
         f"| val OA | {m.get('OA', float('nan')):.4f} |",
         f"| val kappa | {m.get('kappa', float('nan')):.4f} |", "",
         "## Per-class (validation, 136 real fold-1 tiles)", "",
         "| class | IoU | UA (prec) | PA (rec) |", "|---|---|---|---|"]
    for n in CLASS_NAMES:
        L.append(f"| {n} | {m['IoU'][n]:.4f} | {m['UA'][n]:.4f} | {m['PA'][n]:.4f} |")
    L += ["", "Validation contains **no synthetic pixels** in either arm.", "",
          "Not comparable to the part-1 fold-1 run: that one used a 260-tile train split, had "
          "frozen augmentation (the persistent_workers bug), determinism on, and an epoch-based "
          "schedule."]
    p = Path(cfg.OUT_DIR) / "README.md"; p.write_text("\n".join(L))
    hf_push(p, "README.md", f"card step {step}")

print("HF ready:", HF_REPO if api else "(disabled)")

## 17. Weights & Biases

`resume="allow"` with a deterministic run id derived from `RUN_NAME`, so a resumed session continues
the same run instead of starting a second one next to it.

In [ ]:
WB = None
if cfg.WANDB:
    try:
        import wandb
        from kaggle_secrets import UserSecretsClient
        wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
        WB = wandb.init(project=cfg.WANDB_PROJECT, name=RUN_NAME, id=RUN_NAME.replace("/", "-"),
                        resume="allow",
                        config={**asdict(cfg), **EFF, "n_train_rows": len(TRAIN_IDX),
                                "n_real": len(REAL_TRAIN_IDX), "n_fake": N_FAKE_LOADED,
                                "train_sha1": split_sha1, "arch_sig": ARCH_SIG})
        print("wandb:", WB.url)
    except Exception as e:
        print("wandb disabled:", str(e)[:160]); WB = None

## 18. Train / validate loops

`batch_stream()` is an endless iterator over the train loader. It rebuilds the loader's iterator
whenever it runs dry and counts passes for logging, which is the only bookkeeping the step-based
loop needs — augmentation no longer depends on knowing which pass it is on. `criterion.set_step()`
is called **every step**, so the Lovasz ramp and the clDice switch-on are driven by the global step
counter and survive a resume.

`scaler.unscale_(optimizer)` runs **before** `clip_grad_norm_`. Clipping loss-scaled gradients would
make the 1.0 threshold meaningless — this is the exact bug that let 6e5 gradient norms through in
early part-1 runs. The pre-clip norm is logged as the direct check against that.

Validation runs on the **EMA** weights and accumulates two confusion matrices, full and core. It
covers all 408 real val images every time; at 200-step intervals that is 30 validations over a full
run, a few minutes total, and it buys a properly resolved early-stopping curve.

In [ ]:
PASS_COUNT = {"n": 0}

def batch_stream():
    while True:
        for b in train_dl:
            yield b
        PASS_COUNT["n"] += 1

def train_steps(stream, n_steps, step0):
    model.train()
    agg = dict(loss=0.0, ce=0.0, lovasz=0.0, cldice=0.0); gn_acc = []
    optimizer.zero_grad(set_to_none=True)
    for k in range(n_steps):
        criterion.set_step(step0 + k)
        x, y, w = next(stream)
        x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        y = y.to(DEVICE, non_blocking=True); w = w.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=cfg.AMP):
            logits = model(x)
        loss, parts = criterion(logits, y, w)                 # fp32, outside autocast
        scaler.scale(loss / cfg.GRAD_ACCUM).backward()
        if (k + 1) % cfg.GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)                        # MUST precede clipping
            gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.CLIP_NORM)
            gn_acc.append(float(gn))
            prev_scale = scaler.get_scale()
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True); scheduler.step()
            if ema is not None and scaler.get_scale() >= prev_scale:
                ema.update(model)
        agg["loss"] += float(loss); agg["ce"] += parts["ce"]
        agg["lovasz"] += parts["lovasz"]; agg["cldice"] += parts["cldice"]
        if k % 50 == 0:
            print(f"  s{step0+k} loss {float(loss):.4f} (ce {parts['ce']:.4f} "
                  f"lov {parts['lovasz']:.4f} cld {parts['cldice']:.4f}) "
                  f"lam {parts['lam']:.2f} mu {parts['mu']:.2f} "
                  f"gnorm {gn_acc[-1] if gn_acc else 0:.3f}", flush=True)
    n = max(n_steps, 1)
    return ({"train_loss": agg["loss"]/n, "train_ce": agg["ce"]/n,
             "train_lovasz": agg["lovasz"]/n, "train_cldice": agg["cldice"]/n,
             "lovasz_lambda": criterion.lam, "cldice_mu": criterion.mu,
             "gate": float(model.gate.detach()) if cfg.USE_DETAIL else 0.0},
            float(np.mean(gn_acc)) if gn_acc else 0.0,
            float(np.max(gn_acc)) if gn_acc else 0.0)

@torch.no_grad()
def validate():
    if ema: ema.apply_to(model)
    model.eval()
    cm_f, cm_c = new_cm(), new_cm()
    vloss, vce, nb = 0.0, 0.0, 0
    for x, y, w in val_dl:
        x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        y = y.to(DEVICE, non_blocking=True); w = w.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=cfg.AMP):
            logits = model(x)
        loss, vparts = criterion(logits, y, w)
        vloss += float(loss); vce += vparts["ce"]; nb += 1
        pred = logits.float().argmax(1)
        update_cm(cm_f, pred, y)
        update_cm(cm_c, pred, y, mask=(w >= cfg.CORE_THRESH))
    if ema: ema.restore(model)
    model.train()
    return ({"val_loss": vloss / max(nb, 1), "val_ce": vce / max(nb, 1)},
            cm_metrics(cm_f), cm_metrics(cm_c))

print("loops ready")

## 19. Main loop with automatic resume

Early stopping is on **validation mIoU**, not val loss — the loss is class-weighted, CSA-weighted and
ramped, and moves for reasons mIoU does not care about. Patience is 8 validations on any improvement,
which is the rule the B0 and UNetFormer Stage-4 notebooks use; part 1's `MIN_DELTA=0.002`
reference-tracking is deliberately dropped so all three Stage-4 architectures share one stopping rule.

`MAX_HOURS` stops the session cleanly and pushes `last.pt`; re-running with the same `ARM` picks up
from that step, including the loss-ramp state, since the ramps are functions of the global step.

**Read CE, not the total loss.** The printed total is `CE + lambda*Lovasz + mu*clDice` and both
coefficients move — the total is non-stationary by construction. Only `train_ce` / `val_ce` compare
across the run. `gen_gap_ce = val_ce - train_ce` is the overfitting watch: widening past 1.0 while
mIoU stalls is the exact signature this dataset produced on B4 and the ResNet-50 U-Net, and it is
the one thing most likely to appear in R1 for a *new* reason — 804 synthetic images whose appearance
distribution is not quite the real one.

In [ ]:
CKPT_DIR = Path(cfg.OUT_DIR); CKPT_DIR.mkdir(parents=True, exist_ok=True)
LAST, BEST = CKPT_DIR / "last.pt", CKPT_DIR / "best.pt"

step, best_miou, best_step, bad, history = 0, -1.0, -1, 0, []
best_metrics, best_core = {}, {}

def _compatible(sd):
    cur = model.state_dict()
    if set(cur) != set(sd):
        return False, (f"key sets differ (checkpoint-only {sorted(set(sd)-set(cur))[:4]}, "
                       f"model-only {sorted(set(cur)-set(sd))[:4]})")
    for k in cur:
        if tuple(cur[k].shape) != tuple(sd[k].shape):
            return False, f"shape mismatch at {k}: {tuple(sd[k].shape)} vs {tuple(cur[k].shape)}"
    return True, ""

def _purge_stale(reason):
    print(f"!! IGNORING the remote checkpoint -- {reason}")
    print(f"   This arm starts from scratch; stale artifacts are being deleted from {HF_REPO}.")
    if api and cfg.HF_PUSH:
        for f in ("last.pt", "best.pt"):
            try: api.delete_file(path_in_repo=f, repo_id=HF_REPO)
            except Exception: pass

remote_last = hf_try_download("last.pt") if cfg.RESUME else None
if remote_last:
    ck = torch.load(remote_last, map_location="cpu", weights_only=False)
    ok, why = _compatible(ck["model"])
    if ck.get("arch") != cfg.ARCH_VERSION or ck.get("arch_sig") != ARCH_SIG:
        _purge_stale(f'architecture {ck.get("arch","<pre-versioning>")!r}/{ck.get("arch_sig")} '
                     f'!= {cfg.ARCH_VERSION!r}/{ARCH_SIG}')
    elif ck.get("arm") != cfg.ARM:
        _purge_stale(f'checkpoint is arm {ck.get("arm")!r}, this session is {cfg.ARM!r}')
    elif not ok:
        _purge_stale(why)
    else:
        model.load_state_dict(ck["model"], strict=True); optimizer.load_state_dict(ck["optim"])
        scaler.load_state_dict(ck["scaler"]); scheduler.load_state_dict(ck["sched"])
        if ema and ck.get("ema"):
            ema.shadow = {k: v.to(DEVICE) for k, v in ck["ema"].items()}
        step = ck["step"]; best_miou = ck["best_miou"]; best_step = ck["best_step"]
        bad = ck.get("bad", 0); history = ck.get("history", [])
        print(f"RESUMED at step {step} (best mIoU {best_miou:.4f} @ s{best_step}, bad {bad})")
elif cfg.RESUME:
    print("no remote checkpoint -- starting fresh")
else:
    print("RESUME=False -- starting fresh, remote checkpoint ignored")

torch.cuda.reset_peak_memory_stats()
t_start, stream = time.time(), batch_stream()
n_val, STOP = 0, None

while step < cfg.MAX_STEPS:
    n = min(cfg.VAL_EVERY_STEPS, cfg.MAX_STEPS - step)
    t0 = time.time()
    tr, gn_mean, gn_max = train_steps(stream, n, step)
    step += n; n_val += 1
    vl, mfull, mcore = validate()
    peak_gb = torch.cuda.max_memory_allocated() / 1e9
    row = {"step": step, "pass": PASS_COUNT["n"], **tr, **vl,
           "gen_gap_ce": vl["val_ce"] - tr["train_ce"],
           "val_mIoU": mfull["mIoU"], "val_mF1": mfull["mF1"], "val_OA": mfull["OA"],
           "val_kappa": mfull["kappa"], "val_fwIoU": mfull["fwIoU"], "core_mIoU": mcore["mIoU"],
           "grad_norm_mean": gn_mean, "grad_norm_max": gn_max,
           "lr_enc": scheduler.get_last_lr()[LR_IDX_ENC],
           "lr_head": scheduler.get_last_lr()[LR_IDX_HEAD],
           "lr_new": scheduler.get_last_lr()[LR_IDX_NEW],
           "peak_vram_gb": peak_gb, "block_sec": time.time() - t0,
           **{f"IoU/{k}": v for k, v in mfull["IoU"].items()},
           **{f"PA/{k}": v for k, v in mfull["PA"].items()},
           **{f"UA/{k}": v for k, v in mfull["UA"].items()}}
    history.append(row)
    if WB: WB.log(row, step=step)
    print(f'[s{step}] CE {tr["train_ce"]:.3f}/{vl["val_ce"]:.3f} (gap {row["gen_gap_ce"]:+.2f}) '
          f'| mIoU {mfull["mIoU"]:.4f} (core {mcore["mIoU"]:.4f}) '
          f'| road {mfull["IoU"]["Road"]:.3f} river {mfull["IoU"]["River"]:.3f} '
          f'| gate {tr["gate"]:.3f} gn {gn_mean:.2f} | {row["block_sec"]:.0f}s '
          f'| vram {peak_gb:.1f}GB', flush=True)
    if step > 1500 and row["gen_gap_ce"] > 1.0 and mfull["mIoU"] < best_miou:
        print("   ^ CE generalisation gap widening while mIoU stalls -- overfitting signature")

    if mfull["mIoU"] > best_miou:
        best_miou, best_step, bad = mfull["mIoU"], step, 0
        best_metrics, best_core = mfull, mcore
        # EMA shadow holds only float tensors -- merge into the full state dict so a later
        # load_state_dict(strict=True) does not trip over int buffers.
        sd = {k: v.detach().clone() for k, v in model.state_dict().items()}
        if ema:
            for k, v in ema.shadow.items(): sd[k] = v.to(sd[k].dtype)
        torch.save({"model": {k: v.cpu() for k, v in sd.items()}, "cfg": asdict(cfg),
                    "arch": cfg.ARCH_VERSION, "arch_sig": ARCH_SIG, "arm": cfg.ARM,
                    "class_names": CLASS_NAMES, "metrics_full": mfull, "metrics_core": mcore,
                    "step": step, "efficiency": EFF}, BEST)
        hf_push(BEST, "best.pt", f"best mIoU {best_miou:.4f} @ s{step}")
        push_model_card(mfull, step)
        print(f"  -> new best {best_miou:.4f}, pushed")
    else:
        bad += 1

    elapsed_h = (time.time() - t_start) / 3600
    if bad >= cfg.PATIENCE_VALS: STOP = "early stop (patience)"
    elif elapsed_h > cfg.MAX_HOURS: STOP = "session budget"
    elif step >= cfg.MAX_STEPS: STOP = "max steps"

    if (n_val % cfg.PUSH_LAST_EVERY_VALS == 0) or STOP:
        torch.save({"model": model.state_dict(), "optim": optimizer.state_dict(),
                    "scaler": scaler.state_dict(), "sched": scheduler.state_dict(),
                    "ema": (ema.shadow if ema else None), "step": step, "bad": bad,
                    "best_miou": best_miou, "best_step": best_step, "history": history,
                    "cfg": asdict(cfg), "arch": cfg.ARCH_VERSION, "arch_sig": ARCH_SIG,
                    "arm": cfg.ARM}, LAST)
        hf_push(LAST, "last.pt", f"state s{step}")

    if STOP:
        print(f"STOP: {STOP} at step {step} (best {best_miou:.4f} @ s{best_step})")
        break

TRAIN_HOURS = (time.time() - t_start) / 3600
print(f"session done | best mIoU {best_miou:.4f} @ step {best_step} | {TRAIN_HOURS:.2f} h")
if STOP == "session budget":
    print("RE-RUN THIS NOTEBOOK WITH THE SAME ARM -- it resumes from last.pt")

## 20. Final evaluation — full, core, clDice, per tile, bootstrap

The best checkpoint is reloaded and evaluated once, properly. Four things happen here.

**Full protocol** — every valid pixel. This is the number that goes in the Stage-4 table.

**Core protocol** — pixels with `relw >= CORE_THRESH`, outside the unreliable boundary band. The gap
between the two separates model error from annotation error. Note that the core mask is derived from
the *real* validation masks in both arms, so the core comparison is untouched by the synthetic CSA
question in section 9.

**Per-base-tile confusion matrices.** The 136 val tiles each get their own 6x6 matrix, summed over
their three flip variants. Summing all 136 reproduces the pooled result exactly, so nothing is lost —
but keeping them separate is what lets section 22 bootstrap the R1-R0 **difference** on paired
resamples, a much tighter statistic than comparing two independent intervals.

**clDice** for Road and River by true skeletonisation. Images where the class is absent from both GT
and prediction are skipped rather than scored 1.0, which would badly inflate the River number.

In [ ]:
_bp = BEST if BEST.exists() else None
if _bp is None:
    _bp = hf_try_download("best.pt")
    assert _bp, "no best.pt locally or on the Hub -- training never completed a validation"
    print("best.pt pulled from the Hub")
_ck = torch.load(_bp, map_location="cpu", weights_only=False)
assert _ck.get("arch_sig") == ARCH_SIG, "best.pt architecture does not match this notebook"
best_model = build_model()
best_model.load_state_dict({k: v.to(DEVICE) for k, v in _ck["model"].items()}, strict=True)
best_model.eval()
BEST_STEP = _ck.get("step", best_step)

VAL_TILE_IDS = sorted(va_tiles)
tile_pos = {b: i for i, b in enumerate(VAL_TILE_IDS)}
TILE_CM = np.zeros((len(VAL_TILE_IDS), cfg.NUM_CLASSES, cfg.NUM_CLASSES), dtype=np.int64)
cm_full, cm_core = new_cm(), new_cm()
cld = {1: [], 2: []}
per_image = []
peak_vram = torch.cuda.max_memory_allocated() / 2**30

with torch.no_grad():
    for j0 in tqdm(range(0, len(VAL_IDX), cfg.BATCH_SIZE), desc="final eval"):
        pos = list(range(j0, min(j0 + cfg.BATCH_SIZE, len(VAL_IDX))))
        xs, ys, ws = zip(*[val_ds[p] for p in pos])
        x = torch.stack(xs).to(DEVICE).to(memory_format=torch.channels_last)
        y = torch.stack(ys).to(DEVICE); w = torch.stack(ws).to(DEVICE)
        with torch.cuda.amp.autocast(enabled=cfg.AMP):
            logits = best_model(x)
        pred = logits.float().argmax(1)
        update_cm(cm_full, pred, y)
        update_cm(cm_core, pred, y, mask=(w >= cfg.CORE_THRESH))
        pn, yn = pred.cpu().numpy(), y.cpu().numpy()
        for b, p in enumerate(pos):
            row = int(VAL_IDX[p]); base = int(TILE_OF[row])
            c1 = new_cm(); update_cm(c1, pred[b:b+1], y[b:b+1])
            TILE_CM[tile_pos[base]] += c1.cpu().numpy().astype(np.int64)
            per_image.append({"row": row, "tile": base, "mIoU": cm_metrics(c1)["mIoU"]})
            valid = yn[b] != cfg.IGNORE_INDEX
            for ci in (1, 2):
                v = cldice_pair((pn[b] == ci) & valid, (yn[b] == ci) & valid)
                if v is not None: cld[ci].append(v)

M_FULL, M_CORE = cm_metrics(cm_full), cm_metrics(cm_core)
CLDICE = {CLASS_NAMES[ci]: (float(np.mean(cld[ci])) if cld[ci] else None) for ci in (1, 2)}
CLDICE_N = {CLASS_NAMES[ci]: len(cld[ci]) for ci in (1, 2)}

# the per-tile matrices must reproduce the pooled one exactly, or the bootstrap is on sand
assert np.array_equal(TILE_CM.sum(0), cm_full.cpu().numpy().astype(np.int64)), \
    "per-tile confusion matrices do not sum to the pooled matrix"

def boot_metrics(tile_cm, n_boot, seed=0):
    rs = np.random.RandomState(seed)
    N = tile_cm.shape[0]
    mious, per_cls = [], []
    for _ in range(n_boot):
        pick = rs.randint(0, N, N)
        cm = tile_cm[pick].sum(0).astype(np.float64)
        tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
        den = tp + fp + fn
        iou = np.where(den > 0, tp / np.maximum(den, 1e-12), np.nan)   # absent class -> excluded
        mious.append(float(np.nanmean(iou))); per_cls.append(iou)
    return np.array(mious), np.array(per_cls)

BM, BPC = boot_metrics(TILE_CM, cfg.BOOTSTRAP_N, cfg.SEED)
BOOT = {"n": cfg.BOOTSTRAP_N, "unit": "val base tile (136)",
        "mIoU_mean": float(BM.mean()),
        "mIoU_ci95": [float(np.percentile(BM, 2.5)), float(np.percentile(BM, 97.5))],
        "per_class_ci95": {n: [float(np.nanpercentile(BPC[:, i], 2.5)),
                               float(np.nanpercentile(BPC[:, i], 97.5))]
                           for i, n in enumerate(CLASS_NAMES)}}

print(f'\narm {cfg.ARM} | FULL mIoU {M_FULL["mIoU"]:.4f} '
      f'[{BOOT["mIoU_ci95"][0]:.4f}, {BOOT["mIoU_ci95"][1]:.4f}] | mF1 {M_FULL["mF1"]:.4f} '
      f'| OA {M_FULL["OA"]:.4f} | fwIoU {M_FULL["fwIoU"]:.4f} | kappa {M_FULL["kappa"]:.4f}')
print(f'CORE mIoU {M_CORE["mIoU"]:.4f}  (gap = annotation-noise contribution: '
      f'{M_CORE["mIoU"]-M_FULL["mIoU"]:+.4f})')
print("clDice:", {k: (round(v, 4) if v is not None else None) for k, v in CLDICE.items()},
      "| n images", CLDICE_N)
print(f'\n{"class":<13}{"IoU":>8}{"95% CI":>20}{"UA":>8}{"PA":>8}{"coreIoU":>9}')
for n in CLASS_NAMES:
    lo, hi = BOOT["per_class_ci95"][n]
    print(f'{n:<13}{M_FULL["IoU"][n]:>8.4f}   [{lo:.4f}, {hi:.4f}]'
          f'{M_FULL["UA"][n]:>8.4f}{M_FULL["PA"][n]:>8.4f}{M_CORE["IoU"][n]:>9.4f}')

## 21. Persist results

`tile_cm_{arm}.npy` is the file section 22 needs from **both** arms. Everything else mirrors the
part-1 and Stage-4 B0 outputs so the result files stay diffable across models and stages.

In [ ]:
import pandas as pd, csv
EFF["peak_train_VRAM_GB"] = peak_vram
EFF["train_hours_this_session"] = TRAIN_HOURS

RESULT = {"run": RUN_NAME, "model": "TC-SegFormer", "arm": cfg.ARM, "backbone": cfg.BACKBONE,
          "split_mode": cfg.SPLIT_MODE, "fold": cfg.FOLD, "seed": cfg.SEED,
          "best_step": BEST_STEP, "n_train_rows": len(TRAIN_IDX),
          "n_real": len(REAL_TRAIN_IDX), "n_fake": N_FAKE_LOADED,
          "n_train_tiles": len(tr_tiles), "n_val_tiles": len(va_tiles),
          "train_sha1": split_sha1, "arch_sig": ARCH_SIG,
          "components": {"detail": cfg.USE_DETAIL, "csa": cfg.CSA_ENABLE,
                         "csa_on_fake": cfg.CSA_ON_FAKE, "cldice": cfg.CLDICE_ENABLE,
                         "balanced_sampler": cfg.BALANCED_SAMPLER,
                         "weights_from": cfg.WEIGHTS_FROM,
                         "sampler_freq_from": cfg.SAMPLER_FREQ_FROM,
                         "aug_flip_fake": cfg.AUG_FLIP_FAKE},
          "full": M_FULL, "core": M_CORE, "clDice": CLDICE, "clDice_n_images": CLDICE_N,
          "bootstrap": BOOT, "efficiency": EFF,
          "class_weights": CLASS_W.tolist(), "beta": BETA}

jp = Path(cfg.OUT_DIR) / f"results_{tag}.json"; json.dump(RESULT, open(jp, "w"), indent=2, default=float)
np.save(Path(cfg.OUT_DIR) / f"tile_cm_{cfg.ARM}.npy", TILE_CM)
json.dump({"val_tile_ids": VAL_TILE_IDS, "arm": cfg.ARM, "train_sha1": split_sha1},
          open(Path(cfg.OUT_DIR) / f"tile_index_{cfg.ARM}.json", "w"), indent=2)

flat = {"model": "TC-SegFormer", "arm": cfg.ARM, "split_mode": cfg.SPLIT_MODE, "fold": cfg.FOLD,
        "seed": cfg.SEED, "n_train": len(TRAIN_IDX), "n_fake": N_FAKE_LOADED,
        "best_step": BEST_STEP,
        "mIoU": M_FULL["mIoU"], "mIoU_lo": BOOT["mIoU_ci95"][0], "mIoU_hi": BOOT["mIoU_ci95"][1],
        "core_mIoU": M_CORE["mIoU"], "mF1": M_FULL["mF1"], "OA": M_FULL["OA"],
        "fwIoU": M_FULL["fwIoU"], "kappa": M_FULL["kappa"],
        **{f"IoU_{k}": v for k, v in M_FULL["IoU"].items()},
        **{f"UA_{k}": v for k, v in M_FULL["UA"].items()},
        **{f"PA_{k}": v for k, v in M_FULL["PA"].items()},
        **{f"clDice_{k}": v for k, v in CLDICE.items()},
        "params_M": EFF["params_M"], "GFLOPs_512": EFF.get("GFLOPs_512"),
        "fps_bs1": EFF["infer_FPS_bs1"], "peak_VRAM_GB": peak_vram,
        "train_hours": TRAIN_HOURS}
cp = Path(cfg.OUT_DIR) / f"results_{tag}.csv"; pd.DataFrame([flat]).to_csv(cp, index=False)
hp = Path(cfg.OUT_DIR) / f"history_{tag}.csv"; pd.DataFrame(history).to_csv(hp, index=False)

for p, n in [(jp, jp.name), (cp, cp.name), (hp, hp.name),
             (Path(cfg.OUT_DIR) / f"tile_cm_{cfg.ARM}.npy", f"tile_cm_{cfg.ARM}.npy"),
             (Path(cfg.OUT_DIR) / f"tile_index_{cfg.ARM}.json", f"tile_index_{cfg.ARM}.json")]:
    hf_push(p, n, "results")
if WB: WB.summary.update(flat)
print(pd.DataFrame([flat]).T.to_string())
print("\nresults pushed to", HF_REPO)

## 22. R0 vs R1 — the actual Stage-4 result

**Run this only after both arms have finished.** It pulls both result sets from the Hub and computes
the comparison the paper reports.

The headline is a **paired bootstrap**: each of the 2000 resamples draws the same 136 tile indices
for both arms and takes the difference, so tile-to-tile difficulty cancels instead of two
independent variances adding. The p-value is the fraction of resamples in which the difference has
the opposite sign to the observed one — a two-sided sign test over the resample distribution, not a
t-test, with no normality assumption.

One thing this cannot tell you: both arms ran a single seed, so the interval covers sampling
variation across validation tiles, not variation across training runs. State that in Limitations
rather than letting the interval imply more than it measures.

In [ ]:
def pull_arm(arm):
    repo = f"{cfg.HF_USER}/{cfg.HF_PREFIX}-{arm}-{cfg.SPLIT_MODE}-fold{cfg.FOLD}"
    t = f"{arm}_{cfg.SPLIT_MODE}_fold{cfg.FOLD}"
    res = json.load(open(hf_hub_download(repo, f"results_{t}.json", token=HF_TOKEN)))
    cm  = np.load(hf_hub_download(repo, f"tile_cm_{arm}.npy", token=HF_TOKEN))
    idx = json.load(open(hf_hub_download(repo, f"tile_index_{arm}.json", token=HF_TOKEN)))
    return res, cm, idx

try:
    R0, CM0, IX0 = pull_arm("R0")
    R1, CM1, IX1 = pull_arm("R1")
except Exception as e:
    print(f"both arms not finished yet ({type(e).__name__}) -- run this after the second one")
    R0 = None

if R0 is not None:
    assert IX0["val_tile_ids"] == IX1["val_tile_ids"], "arms validated on different tiles"
    assert IX0["train_sha1"] == IX1["train_sha1"], "arms used different real splits"
    N = CM0.shape[0]

    def iou_from(cm):
        tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
        den = tp + fp + fn
        return np.where(den > 0, tp / np.maximum(den, 1e-12), np.nan)

    rs = np.random.RandomState(cfg.SEED)
    d_miou, d_cls = [], []
    for _ in range(cfg.BOOTSTRAP_N):
        pick = rs.randint(0, N, N)                       # SAME tiles for both arms
        i0 = iou_from(CM0[pick].sum(0).astype(np.float64))
        i1 = iou_from(CM1[pick].sum(0).astype(np.float64))
        d_miou.append(np.nanmean(i1) - np.nanmean(i0)); d_cls.append(i1 - i0)
    d_miou = np.array(d_miou); d_cls = np.array(d_cls)

    obs = R1["full"]["mIoU"] - R0["full"]["mIoU"]
    p = min(1.0, float(np.mean(d_miou <= 0) if obs > 0 else np.mean(d_miou >= 0)) * 2)
    print(f"{'metric':<16}{'R0':>10}{'R1':>10}{'delta':>10}{'95% CI on delta':>26}")
    print(f"{'mIoU':<16}{R0['full']['mIoU']:10.4f}{R1['full']['mIoU']:10.4f}{obs:+10.4f}"
          f"    [{np.percentile(d_miou,2.5):+.4f}, {np.percentile(d_miou,97.5):+.4f}]")
    for k in ["mF1", "OA", "fwIoU", "kappa"]:
        print(f"{k:<16}{R0['full'][k]:10.4f}{R1['full'][k]:10.4f}{R1['full'][k]-R0['full'][k]:+10.4f}")
    print(f"{'core mIoU':<16}{R0['core']['mIoU']:10.4f}{R1['core']['mIoU']:10.4f}"
          f"{R1['core']['mIoU']-R0['core']['mIoU']:+10.4f}")
    print(f"\npaired-bootstrap two-sided p on mIoU: {p:.4f}  (n={cfg.BOOTSTRAP_N} resamples)")

    print(f"\n{'class':<14}{'IoU R0':>9}{'IoU R1':>9}{'delta':>9}{'95% CI on delta':>26}")
    rows = []
    for i, nm in enumerate(CLASS_NAMES):
        a, b = R0["full"]["IoU"][nm], R1["full"]["IoU"][nm]
        lo, hi = np.nanpercentile(d_cls[:, i], 2.5), np.nanpercentile(d_cls[:, i], 97.5)
        star = " *" if (lo > 0 or hi < 0) else ""
        print(f"{nm:<14}{a:9.4f}{b:9.4f}{b-a:+9.4f}    [{lo:+.4f}, {hi:+.4f}]{star}")
        rows.append({"class": nm, "IoU_R0": a, "IoU_R1": b, "delta": b - a,
                     "ci_lo": float(lo), "ci_hi": float(hi),
                     "UA_R0": R0["full"]["UA"][nm], "UA_R1": R1["full"]["UA"][nm],
                     "PA_R0": R0["full"]["PA"][nm], "PA_R1": R1["full"]["PA"][nm]})
    print("\n* = 95% CI on the difference excludes zero")
    print("\nUA/PA is where the mechanism shows: if River IoU moved because the model finally")
    print("predicts River at all, PA rises. If it moved because it stopped over-predicting, UA rises.")

    print(f"\n{'clDice':<14}{'R0':>9}{'R1':>9}{'delta':>9}")
    for k in R0["clDice"]:
        a, b = R0["clDice"][k], R1["clDice"][k]
        if a is not None and b is not None:
            print(f"{k:<14}{a:9.4f}{b:9.4f}{b-a:+9.4f}")

    comp = {"model": "TC-SegFormer",
            "mIoU_R0": R0["full"]["mIoU"], "mIoU_R1": R1["full"]["mIoU"], "delta_mIoU": obs,
            "delta_mIoU_ci95": [float(np.percentile(d_miou, 2.5)),
                                float(np.percentile(d_miou, 97.5))],
            "p_two_sided": p, "n_boot": cfg.BOOTSTRAP_N,
            "core_mIoU_R0": R0["core"]["mIoU"], "core_mIoU_R1": R1["core"]["mIoU"],
            "per_class": rows,
            "best_step": {"R0": R0["best_step"], "R1": R1["best_step"]}}
    json.dump(comp, open(f"{cfg.OUT_DIR}/stage4_tc_R0_vs_R1.json", "w"), indent=2, default=float)
    pd.DataFrame(rows).to_csv(f"{cfg.OUT_DIR}/stage4_tc_R0_vs_R1.csv", index=False)
    hf_push(f"{cfg.OUT_DIR}/stage4_tc_R0_vs_R1.json", "stage4_tc_R0_vs_R1.json", "arm comparison")
    hf_push(f"{cfg.OUT_DIR}/stage4_tc_R0_vs_R1.csv", "stage4_tc_R0_vs_R1.csv", "arm comparison")

## 23. Standalone inference from Hugging Face

Self-contained: paste into any notebook with `transformers` and a token. It rebuilds the
architecture from the config embedded in `best.pt`, so the checkpoint alone is sufficient.

**This fixes a real bug in part 1's section 23.** That version named the H/2 branch's final conv
`s.classifier` while the trained model names it `self.refine`, so `load_state_dict(strict=True)`
raised `Missing key(s): refine.weight` on every genuine checkpoint. The name here matches the
training module, and the load is asserted rather than hoped for.

In [ ]:
# ===================== STANDALONE -- paste into an empty notebook =====================
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, cv2
from huggingface_hub import hf_hub_download
from transformers import SegformerConfig, SegformerForSemanticSegmentation

REPO = "sugam24/geonusaf-s4-tcsegformer-R1-block-fold1"   # <-- or ...-R0-...
DEV = "cuda" if torch.cuda.is_available() else "cpu"

class _Detail(nn.Module):
    def __init__(s, chs):
        super().__init__(); c1, c2, c3 = chs
        b = lambda i, o, st: nn.Sequential(nn.Conv2d(i, o, 3, st, 1, bias=False),
                                           nn.GroupNorm(min(8, o), o), nn.ReLU(inplace=True))
        s.net = nn.Sequential(b(3, c1, 2), b(c1, c2, 1), b(c2, c3, 1)); s.out_ch = c3
    def forward(s, x): return s.net(x)

class _TC(nn.Module):
    def __init__(s, C):
        super().__init__()
        conf = SegformerConfig.from_pretrained(C["BACKBONE"]); conf.num_labels = C["NUM_CLASSES"]
        base = SegformerForSemanticSegmentation.from_pretrained(
            C["BACKBONE"], config=conf, ignore_mismatched_sizes=True)
        s.encoder, s.head = base.segformer, base.decode_head
        s.use_detail = C["USE_DETAIL"]
        if s.use_detail:
            s.proj = nn.Conv2d(conf.decoder_hidden_size, C["FUSE_CH"], 1)
            s.detail = _Detail(tuple(C["DETAIL_CH"]))
            s.gate = nn.Parameter(torch.tensor(float(C["DETAIL_GATE_INIT"])))
            s.fuse = nn.Sequential(
                nn.Conv2d(C["FUSE_CH"] + s.detail.out_ch, C["FUSE_CH"], 3, 1, 1, bias=False),
                nn.GroupNorm(8, C["FUSE_CH"]), nn.ReLU(inplace=True))
            s.refine = nn.Conv2d(C["FUSE_CH"], C["NUM_CLASSES"], 1)   # <-- 'refine', not 'classifier'
    def forward(s, x):
        H, W = x.shape[-2:]
        hs = s.encoder(x, output_hidden_states=True, return_dict=True).hidden_states
        mlps = getattr(s.head, "linear_c", None) or s.head.linear_projections
        outs = []
        for h_, mlp in zip(hs, mlps):
            hh, ww = h_.shape[2], h_.shape[3]
            t = mlp(h_).permute(0, 2, 1).reshape(h_.shape[0], -1, hh, ww)
            outs.append(F.interpolate(t, size=hs[0].shape[2:], mode="bilinear", align_corners=False))
        fused = s.head.dropout(s.head.activation(s.head.batch_norm(
            s.head.linear_fuse(torch.cat(outs[::-1], 1)))))
        logits = F.interpolate(s.head.classifier(fused), size=(H, W), mode="bilinear", align_corners=False)
        if not s.use_detail:
            return logits
        f = F.interpolate(s.proj(fused), size=(H // 2, W // 2), mode="bilinear", align_corners=False)
        f = torch.cat([f, s.gate * s.detail(x)], 1)
        return logits + F.interpolate(s.refine(s.fuse(f)), size=(H, W), mode="bilinear", align_corners=False)

def load_from_hf(repo_id, token=None, device=DEV):
    ck = torch.load(hf_hub_download(repo_id, "best.pt", token=token),
                    map_location="cpu", weights_only=False)
    net = _TC(ck["cfg"])
    missing, unexpected = net.load_state_dict(ck["model"], strict=False)
    assert not missing and not unexpected, f"missing {missing[:4]} unexpected {unexpected[:4]}"
    return net.to(device).eval(), ck["cfg"], ck["metrics_full"], ck.get("step")

MEAN = np.array([0.485, 0.456, 0.406], np.float32); STD = np.array([0.229, 0.224, 0.225], np.float32)
# class index -> RGB (the stored palette is BGR, so it is reversed here)
RGB_PALETTE = np.array([[0,0,128],[0,128,0],[128,0,0],[128,128,0],[0,128,128],[128,0,128],[0,0,0]], np.uint8)

@torch.no_grad()
def predict(net, C, path, size=None):
    size = size or C["IMG_SIZE"]
    im = cv2.imread(str(path), cv2.IMREAD_COLOR)
    im = cv2.resize(im, (size, size), interpolation=cv2.INTER_AREA)
    x = ((im[:, :, ::-1].astype(np.float32) / 255.0) - MEAN) / STD
    x = torch.from_numpy(x.transpose(2, 0, 1))[None].to(DEV)
    with torch.cuda.amp.autocast(enabled=(DEV == "cuda")):
        p = net(x).float().argmax(1)[0].cpu().numpy().astype(np.uint8)
    return p, RGB_PALETTE[p]
# =====================================================================================

inf_net, inf_cfg, inf_metrics, inf_step = load_from_hf(HF_REPO, token=HF_TOKEN)
print(f"loaded {HF_REPO} | val mIoU {inf_metrics['mIoU']:.4f} | step {inf_step}")

## 24. Qualitative triptychs and the confusion matrix

Ranked by per-image mIoU, not sampled at random — the failure cases are what reviewers want, and on
this dataset they are diagnostic: worst tiles dominated by River/UnusedLand confusion is a spectral
problem (dry gravel bars are RGB-identical to water and there is no NIR band) that no architectural
change fixes.

Run this section in **both** sessions and diff the pairs by filename; the tiles selected are this
arm's worst cases, so the R0/R1 comparison you actually want is the same tile under both.

The palette is reversed from the stored BGR before display. Part 1's section 24 passed the BGR
tuples straight to matplotlib, so its figures show Residential in red rather than navy — cosmetic,
but the two sets of figures will not match unless you know why.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

PALETTE_RGB = np.zeros((256, 3), np.uint8)
for bgr, idx in BGR_TO_CLASS.items():
    PALETTE_RGB[idx] = bgr[::-1]
PALETTE_RGB[255] = (0, 0, 0)
def colorize(lab): return PALETTE_RGB[lab]

pi = sorted(per_image, key=lambda d: d["mIoU"])
picks = [("worst", r) for r in pi[:3]] + [("best", r) for r in pi[-3:][::-1]]

fig, axes = plt.subplots(len(picks), 3, figsize=(13, 4.2 * len(picks)))
axes = np.atleast_2d(axes)
for r_i, (kind, rec) in enumerate(picks):
    j = rec["row"]
    im_u8, gt, _ = read_row(j)
    img = im_u8[:, :, ::-1]
    x = ((img.astype(np.float32) / 255.0) - IMNET_MEAN) / IMNET_STD
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=cfg.AMP):
        pr = best_model(torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1)))[None]
                        .to(DEVICE)).float().argmax(1)[0].cpu().numpy()
    pr_vis = np.where(gt == 255, 255, pr).astype(np.uint8)
    for c_i, (im_, t) in enumerate([(img, f'{kind} - {STEMS[j]}'), (colorize(gt), "ground truth"),
                                    (colorize(pr_vis), f'{cfg.ARM} prediction (mIoU {rec["mIoU"]:.3f})')]):
        axes[r_i, c_i].imshow(im_); axes[r_i, c_i].set_title(t, fontsize=9); axes[r_i, c_i].axis("off")
handles = [Patch(color=PALETTE_RGB[i] / 255, label=n) for i, n in enumerate(CLASS_NAMES)]
handles.append(Patch(color="black", label="ignore/unannotated"))
fig.legend(handles=handles, loc="lower center", ncol=7, fontsize=8, frameon=False)
fig.suptitle(f"TC-SegFormer {cfg.ARM} - {cfg.SPLIT_MODE} fold {cfg.FOLD} - best and worst", fontsize=12)
plt.tight_layout(rect=[0, 0.04, 1, 0.97])
p = Path(cfg.OUT_DIR) / f"qualitative_{tag}.png"
plt.savefig(p, dpi=130, bbox_inches="tight"); plt.show()
hf_push(p, p.name, "qualitative")

cmn = np.array(M_FULL["cm_norm"])
fig, ax = plt.subplots(figsize=(7.2, 6))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(6)); ax.set_yticks(range(6))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right"); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth")
ax.set_title(f"Row-normalized confusion - TC-SegFormer {cfg.ARM}, {cfg.SPLIT_MODE} fold {cfg.FOLD}")
for i in range(6):
    for j in range(6):
        ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if cmn[i, j] > 0.5 else "black")
fig.colorbar(im, fraction=0.046)
plt.tight_layout()
p = Path(cfg.OUT_DIR) / f"confusion_{tag}.png"
plt.savefig(p, dpi=150, bbox_inches="tight"); plt.show()
hf_push(p, p.name, "confusion matrix")
if WB: WB.finish()

## 25. Run checklist

1. Internet **ON**. Secrets `HF_TOKEN` (write) + `WANDB_API_KEY`.
2. Attach **both** inputs: `sugam24/satellite-datasets` and
   `sugam24/geonusaf-fake-pairs-block-fold1`.
3. If you have a part-1 `cache512` saved as a Kaggle dataset, attach it and set `PREBUILT_CACHE` —
   the repack is 25-40 min and there are two sessions here. It is safe to reuse: section 6 asserts
   the file ordering and the resolution, and the CSA parameters are unchanged from part 1.
4. **Session 1:** `ARM = "R0"`. Run top to bottom through section 21, then 23-24. Section 22 will
   print "both arms not finished yet" — expected, skip it.
5. **Session 2:** `ARM = "R1"`. Same run. Section 22 now produces the comparison.
6. If a session hits the 11 h budget, re-run with the same `ARM`; it resumes from `last.pt`.

### Asserts that must pass before you walk away
- Section 5: `channel-order check PASSED`
- Section 6: no `cache was built from a different file ordering`
- Section 8: `four-way agreement with S1/S2/S3 ASSERTED` — if this fails, the real split disagrees
  with what the generators were trained on and the whole experiment is void
- Section 9 (R1 only): 804 synthetic pairs loaded, contract asserts pass, no synthetic row in
  validation, and the `CSA w real` / `CSA w synth` columns are within about 20% of each other
- Section 11: `augmentation liveness check PASSED`
- Section 13: `logits after upsample: (2, 6, 512, 512)`, `params total ~3.82 M`
- Section 20: the per-tile matrices sum to the pooled matrix (asserted)

### Sanity gates
Because `refine` is initialised near zero, step 0 is effectively plain SegFormer-B0, so this run
should track the B0 Stage-4 R0 curve early and then diverge upward.

| Step | Expect |
|---|---|
| 200 | mIoU > 0.15 |
| 1000 | mIoU > 0.30, Road IoU > 0.05, `gate` moved off 0.100 |
| 2000 | within ~0.05 of the B0 Stage-4 R0 curve at the same step |

If it tracks *below* B0 early, the new modules are fighting the pretrained head — lower `LR_NEW`.
If `gate` is still at 0.100 by step 1000 the detail path is receiving no useful gradient, which is a
different failure and worth stopping for.

### Reading the result
Section 22's mIoU delta is this model's Stage-4 headline; the per-class table is where the argument
lives. River and Road are what the synthetic data was supposed to help, and the UA/PA columns say
*how* — whether the model started predicting the class or stopped over-predicting it.

TC-SegFormer is the interesting arm precisely because it already has three mechanisms aimed at rare
thin classes. If synthetic data helps B0 but not TC-SegFormer, the honest reading is that the
synthetic pairs and the CSA/clDice/sampler stack are addressing the same bottleneck and do not
compound — which is a finding, not a failure, and is exactly why all three architectures are being
run rather than just the proposed one.

**A negative delta is a publishable result.** Do not tune the mixing ratio against the validation
set to find a positive number — that is selection on the test set, and with one fold and one seed it
would be indefensible.

### What this run cannot support
One seed, one fold, one mixing ratio (R1 = 50% synthetic). The bootstrap interval covers validation
tile sampling only, not run-to-run variation, which is unmeasured here. Scope every claim to those
four facts.